# 08 · Q3 et Q4 : les retours et leur lien avec les remises

**Projet 4 : Analyse des données de vente et retours** · Sprint 4 · US19 et US21

**Question de recherche Q3** : *quels sont les patterns de retour de produits ?*

**Question de recherche Q4** : *existe-t-il une corrélation entre les remises et le taux de retour ?*

**Objectif de l'énoncé servi par ce notebook** : identifier les déterminants des retours, chiffrer ce qu'ils coûtent sur le dernier exercice et dire s'il existe un levier pour les réduire.

**Le cadre d'analyse** : comme dans `06_q1_facteurs_rentabilite`, les chiffres de **situation** (coût des retours, gain possible) portent sur le dernier exercice, **2021**. Ils sont suivis sur les **trois exercices** : **2019**, l'année de référence où tous les magasins gagnaient de l'argent, **2020** et **2021**. Les **mécanismes** (poids des facteurs, tests) sont mesurés sur les trois années réunies, puis vérifiés par exercice. Les trois canaux sont étudiés, avec l'accent sur les **magasins**, dont l'écart avec la boutique en ligne explique la perte de l'entreprise.

| Étape | Ce qu'on fait | Question posée |
|---|---|---|
| 1. Chargement | Ventes avec variables, table commune ventes et charges, charges, dimensions | |
| 2. Portrait | Fréquence, taille et montant des retours, par exercice et par canal | À quoi ressemble un retour, et qu'est-ce qui a changé depuis 2019 ? |
| 3. Le choix de la mesure | Trois taux de retour possibles, selon la taille des lignes et le canal | Comment mesurer sans se tromper ? |
| 4. Q3 : le poids des facteurs | Khi-deux et V de Cramér pour chaque facteur, sur les trois exercices puis par exercice ; sévérité COVID-19 | Qui retourne quoi, où et quand ? |
| 5. Q3 : âge du produit, gamme, canal | Les angles établis par `06` et `06b`, et leur évolution 2019 → 2021 | Les produits anciens, l'entrée de gamme ou un canal concentrent-ils les retours ? |
| 6. Q3 : cas anormaux | Taux standardisés par produit et par magasin, stabilité d'un exercice à l'autre | Y a-t-il des produits ou des magasins à problème ? |
| 7. Q3 : prix et retours | Fréquence des retours selon le prix unitaire | Retourne-t-on plus ce qui est cher ? |
| 8. Q4 : promotions, remises et retours | Comparaison directe, puis régression logistique par canal avec contrôle des facteurs de confusion | La remise augmente-t-elle les retours ? |
| 9. Le coût des retours en 2021 | Marge perdue par canal, face au résultat, magasins, catégories, concentration | Où les retours coûtent-ils, et combien ? |
| 10. Le levier « réduire les retours de moitié » | Vérification du levier de `06`, répartition par canal, variantes ciblées | Que vaut le levier, et peut-on le cibler ? |
| 11. Registre des tests | Correction de Bonferroni | |
| 12. Synthèse | Priorités, réponses à Q3 et Q4, registre des constats, recommandations, points à valider | |

**Lit dans** : `data/clean/fait_ventes_variables.parquet`, `data/clean/fait_charges.parquet`, `data/clean/agregats/ventes_charges_boutique_mois.parquet`, `data/clean/dimensions` (`dim_boutique`, `dim_produit`)

**Écrit dans** : `reports/figures` (figures `08_*.png`) et `reports/livrables/L4_analyse/q3_q4_tableaux.xlsx`

**Utilise** : `src/analyse.py` (khi2, mann_whitney, noter_test, tableau_tests, exporter_tableaux, libellés français)

**Avant de lancer** : les notebooks `00` à `05c` doivent avoir été exécutés sur les ventes complètes.

**Lien avec les autres notebooks** : le levier « réduire les retours de moitié » de `06_q1_facteurs_rentabilite` (section 8) est vérifié ici à partir des lignes de vente (section 10). L'effet des promotions et des remises sur les ventes est traité dans `07_q2_promotions` et `09_q5_remises` ; ici, on regarde seulement leur lien avec les retours.

### Vocabulaire

| Terme | Définition |
|---|---|
| Ligne | le cumul d'un produit, dans une boutique, un jour donné, pour une promotion. Ce n'est pas un ticket de caisse |
| Ligne avec retour | ligne de vente dont au moins une unité a été retournée |
| Fréquence des retours | lignes avec retour ÷ lignes (part des lignes avec retour) |
| Taux de retour en unités | unités retournées ÷ unités vendues |
| Taux de retour en valeur | montant des retours ÷ chiffre d'affaires brut. C'est ce que les retours **coûtent** en chiffre d'affaires |
| Marge perdue | marge qui accompagnait les unités retournées : montant du retour moins le coût de ces unités (`marge_perdue_retours`) |
| Nouveauté de l'année | produit vendu l'année de sa mise en vente (`dim_produit.date_mise_en_vente`), comme dans `06b` |
| Gamme | classe du produit : économique, standard, luxe |
| Magasin ouvert toute l'année | magasin ouvert au 1er janvier de l'exercice (48, 51 puis 53 magasins). Seuls ces magasins sont comparés entre eux |
| V de Cramér | taille d'effet du test du khi-deux, entre 0 et 1. Repères : 0,1 petit, 0,3 moyen, 0,5 grand |
| Rapport de cotes | résultat d'une régression logistique. 1 : pas d'effet. 1,10 : le risque relatif de retour augmente d'environ 10 % |

### Règle de lecture des tests

Avec 715 291 lignes, presque tout écart est « statistiquement significatif ». Chaque test est donc lu sur la valeur p **et** sur la taille de l'effet, et le seuil de 5 % est divisé par le nombre de tests du registre (correction de Bonferroni, section 11).

Chaque constat reçoit un verdict : *retenu* (mesuré), *indice* (compatible, sans preuve directe), *non soutenu* (les données ne montrent pas la trace attendue), *ouvert* (les données ne permettent pas de répondre). Une interprétation métier est signalée « *Interprétation métier (hypothèse)* ».

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf      # régression logistique (fourni avec Anaconda)

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config  -> chemins, années de l'étude (ANNEES_ETUDE, ANNEE_REFERENCE, EXERCICE)
#   viz     -> charte graphique
#   analyse -> khi2 et mann_whitney (tests et tailles d'effet), noter_test et tableau_tests
#              (registre des tests), exporter_tableaux, et les libellés français
from src import config, viz, analyse
from src.analyse import LIBELLES_TYPES, ORDRE_TYPES, NOMS_MOIS, khi2, mann_whitney, noter_test, tableau_tests

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---------------------------------------------------------------------------
# 3. Les trois exercices
# ---------------------------------------------------------------------------
ANNEES = config.ANNEES_ETUDE                  # [2019, 2020, 2021]
REFERENCE, EXERCICE = config.ANNEE_REFERENCE, config.EXERCICE
BASCULE = [a for a in ANNEES if a not in (REFERENCE, EXERCICE)][0]
COULEURS_ANNEES = {REFERENCE: viz.GRIS, BASCULE: viz.CATEGORIES[2], EXERCICE: viz.PRINCIPALE}
COULEURS_TYPES = {"En ligne": viz.CATEGORIES[2], "Revendeur": viz.ACCENT, "Magasins": viz.PRINCIPALE}
print("Racine du projet :", config.ROOT)
print(f"Exercices : {ANNEES} · référence {REFERENCE} · exercice étudié {EXERCICE}")


# ---------------------------------------------------------------------------
# 4. Mise en forme des nombres (virgule française)
# ---------------------------------------------------------------------------
def m_dollars(valeur, decimales=1, signe=False):
    """Montant en millions de dollars, virgule française : 12,3 M$."""
    texte = f"{valeur / 1e6:{'+' if signe else ''},.{decimales}f}".replace(",", " ").replace(".", ",")
    return f"{texte} M$"


def pct(valeur, decimales=1, signe=False):
    """Taux en pourcentage, virgule française : 0.123 -> '12,3 %'."""
    return f"{valeur * 100:{'+' if signe else ''}.{decimales}f} %".replace(".", ",")


def points(valeur, decimales=1):
    """Écart entre deux taux, en points de pourcentage : 0.012 -> '+1,2 point'."""
    return f"{valeur * 100:+.{decimales}f} point".replace(".", ",")


def dec(valeur, decimales=3, signe=True):
    """Coefficient avec virgule française : 0.123 -> '+0,123'."""
    return f"{valeur:{'+' if signe else ''}.{decimales}f}".replace(".", ",")


def nombre(valeur):
    """Entier avec espace des milliers : 12345 -> '12 345'."""
    return f"{valeur:,.0f}".replace(",", " ")


# ---------------------------------------------------------------------------
# 5. Registres
# ---------------------------------------------------------------------------
tableaux = {}      # tableaux à exporter vers Excel à la fin : nom de l'onglet -> tableau
registre = []      # registre des tests d'hypothèse, rempli par noter_test() au fil du notebook
constats = []      # registre des constats, avec leur verdict (comme dans 06b)
ALPHA = 0.05       # seuil de signification avant correction


def constat(texte, mesure, verdict):
    """Ajoute un constat au registre et l'affiche. verdict : retenu, indice, non soutenu ou ouvert."""
    constats.append({"constat": texte, "mesure": mesure, "verdict": verdict})
    print(f"[{verdict:<11}] {texte} · {mesure}")

## 1. Chargement

In [ ]:
chemins = {
    "ventes": config.CLEAN / "fait_ventes_variables.parquet",
    "charges": config.CLEAN / "fait_charges.parquet",
    "commun": config.AGREGATS / "ventes_charges_boutique_mois.parquet",
    "dim_boutique": config.CLEAN_DIMENSIONS / "dim_boutique.parquet",
    "dim_produit": config.CLEAN_DIMENSIONS / "dim_produit.parquet",
}
for nom, chemin in chemins.items():
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord les notebooks 00 à 05c.")

ventes = pd.read_parquet(chemins["ventes"])
charges = pd.read_parquet(chemins["charges"])
commun = pd.read_parquet(chemins["commun"])
dim_boutique = pd.read_parquet(chemins["dim_boutique"])
dim_produit = pd.read_parquet(chemins["dim_produit"])

# ---------------------------------------------------------------------------
# 1. Libellés français
# ---------------------------------------------------------------------------
ventes["type"] = ventes.type_boutique.map(LIBELLES_TYPES)
commun["type"] = commun.type_boutique.map(LIBELLES_TYPES)
reel = charges[charges.scenario == "Réel"].copy()
reel["type"] = reel.type_boutique.map(LIBELLES_TYPES)
ORDRE_GAMMES = ["Économique", "Standard", "Luxe"]
ventes["gamme"] = ventes.classe.map({"Economy": "Économique", "Regular": "Standard", "Deluxe": "Luxe"})

# ---------------------------------------------------------------------------
# 2. Magasin ouvert toute l'année : ouvert au 1er janvier de l'année de la vente
# ---------------------------------------------------------------------------
ouverture = dim_boutique.set_index("id_boutique").date_ouverture
ventes["annee_complete"] = ventes.id_boutique.map(ouverture) < pd.to_datetime(ventes.annee.astype(str) + "-01-02")
magasins_lignes = ventes[(ventes.type == "Magasins") & ventes.annee_complete]
print("Magasins ouverts toute l'année :", magasins_lignes.groupby("annee").id_boutique.nunique().to_dict())

# ---------------------------------------------------------------------------
# 3. Âge du produit (06b) : nouveauté de l'année ou produit plus ancien
# ---------------------------------------------------------------------------
lancement = dim_produit.set_index("id_produit").date_mise_en_vente.dt.year
ventes["annee_de_lancement"] = ventes.id_produit.map(lancement)
ventes["age_produit"] = np.where(ventes.annee <= ventes.annee_de_lancement, "Nouveauté de l'année", "Produit plus ancien")
ORDRE_AGES = ["Nouveauté de l'année", "Produit plus ancien"]
# Âge plus fin, en années écoulées depuis la mise en vente
anciennete = (ventes.annee - ventes.annee_de_lancement).clip(lower=0)
ventes["anciennete"] = pd.cut(anciennete, [-1, 0, 1, 2, 99], labels=["0 (nouveauté)", "1 an", "2 ans", "3 ans et plus"])

print(f"Ventes  : {nombre(len(ventes))} lignes · {ventes.id_boutique.nunique()} boutiques · {ventes.id_produit.nunique()} produits")
print(f"Période : du {ventes.date.min().date()} au {ventes.date.max().date()}")

# Garde-fous : trois années complètes, et une table commune qui porte sur les mêmes ventes
if sorted(ventes.annee.unique()) != ANNEES or ventes.groupby("annee").trimestre.nunique().min() != 4:
    raise ValueError(f"La table des ventes ne couvre pas les années {ANNEES} en entier : relancer les notebooks 00 à 05c.")
if abs(commun.montant_retour.sum() - ventes.montant_retour.sum()) > 1:
    raise ValueError("La table commune ventes et charges ne correspond pas à la table des ventes : relancer 05b_charges.")
if ventes.annee_de_lancement.isna().any():
    raise ValueError("Des produits vendus n'ont pas de date de mise en vente dans dim_produit.")


# ---------------------------------------------------------------------------
# 4. Les mesures de retour par groupe (toutes recalculées à partir des sommes du groupe)
# ---------------------------------------------------------------------------
def retours_par(cle, df=None):
    """Mesures de retour par groupe : lignes, unités par ligne, fréquence, taux en unités et en valeur, montants.

    cle : colonne ou liste de colonnes de regroupement
    """
    df = ventes if df is None else df
    g = df.groupby(cle, observed=True).agg(lignes=("id_vente", "size"),
                                           lignes_avec_retour=("a_retour", "sum"),
                                           unites_vendues=("quantite_vente", "sum"),
                                           unites_retournees=("quantite_retour", "sum"),
                                           ca_brut=("ca_brut", "sum"),
                                           montant_retour=("montant_retour", "sum"),
                                           marge_perdue=("marge_perdue_retours", "sum"))
    g["unites_par_ligne"] = g.unites_vendues / g.lignes
    g["frequence"] = g.lignes_avec_retour / g.lignes
    g["taux_unites"] = g.unites_retournees / g.unites_vendues
    g["taux_valeur"] = g.montant_retour / g.ca_brut
    return g[["lignes", "unites_par_ligne", "frequence", "taux_unites", "taux_valeur", "ca_brut", "montant_retour", "marge_perdue"]]


FORMATS_RETOURS = {"lignes": "{:,.0f}", "unites_par_ligne": "{:.1f}", "frequence": "{:.2%}", "taux_unites": "{:.2%}",
                   "taux_valeur": "{:.2%}", "ca_brut": "{:,.0f}", "montant_retour": "{:,.0f}", "marge_perdue": "{:,.0f}"}


def montrer(table, formats_en_plus=None):
    """Affiche un tableau de retours avec les formats communs."""
    formats = {**FORMATS_RETOURS, **(formats_en_plus or {})}
    display(table.style.format({c: f for c, f in formats.items() if c in table.columns}, na_rep=""))

## 2. Portrait des retours sur les trois exercices

In [ ]:
# ---------------------------------------------------------------------------
# 1. Les ordres de grandeur, trois exercices réunis
# ---------------------------------------------------------------------------
ensemble = retours_par(lambda _: "Ensemble")
montrer(ensemble)
FREQUENCE_GLOBALE = float(ensemble.frequence.iloc[0])       # fréquence de référence pour toute la suite
tableaux["2_ensemble"] = ensemble

# ---------------------------------------------------------------------------
# 2. Combien d'unités sont retournées quand il y a un retour ?
# ---------------------------------------------------------------------------
taille = ventes.quantite_retour.value_counts().sort_index().to_frame("lignes")
taille["part de toutes les lignes"] = taille.lignes / taille.lignes.sum()
avec_retour = taille.lignes.where(taille.index > 0)
taille["part des lignes avec retour"] = avec_retour / avec_retour.sum()
taille.index.name = "unités retournées sur la ligne"
print("Répartition des lignes selon le nombre d'unités retournées :")
display(taille.style.format({"lignes": "{:,.0f}", "part de toutes les lignes": "{:.2%}", "part des lignes avec retour": "{:.1%}"}, na_rep=""))
tableaux["2_taille_des_retours"] = taille

# ---------------------------------------------------------------------------
# 3. Par exercice, et évolutions
# ---------------------------------------------------------------------------
par_annee = retours_par("annee")
montrer(par_annee)
evolution = pd.DataFrame(index=par_annee.columns)
for a, b in [(REFERENCE, BASCULE), (BASCULE, EXERCICE), (REFERENCE, EXERCICE)]:
    # Montants et volumes : évolution relative ; taux : écart en points
    evolution[f"{a} → {b}"] = [par_annee.loc[b, c] - par_annee.loc[a, c] if c in ("frequence", "taux_unites", "taux_valeur")
                               else par_annee.loc[b, c] / par_annee.loc[a, c] - 1 for c in par_annee.columns]
print("Évolutions (taux : écart en points ; autres colonnes : évolution relative) :")
display(evolution.style.format("{:+.2%}"))
tableaux["2_par_exercice"] = par_annee
tableaux["2_evolutions"] = evolution

# ---------------------------------------------------------------------------
# 4. Par canal et par exercice
# ---------------------------------------------------------------------------
par_canal = retours_par(["type", "annee"]).loc[ORDRE_TYPES]
montrer(par_canal)
tableaux["2_par_canal"] = par_canal
for t in ORDRE_TYPES:
    d = par_canal.loc[t]
    print(f"{t:<10} fréquence {pct(d.loc[REFERENCE, 'frequence'])} → {pct(d.loc[EXERCICE, 'frequence'])} · "
          f"taux en valeur {pct(d.loc[REFERENCE, 'taux_valeur'], 2)} → {pct(d.loc[EXERCICE, 'taux_valeur'], 2)} · "
          f"marge perdue {m_dollars(d.loc[REFERENCE, 'marge_perdue'], 2)} → {m_dollars(d.loc[EXERCICE, 'marge_perdue'], 2)} "
          f"({pct(d.loc[EXERCICE, 'marge_perdue'] / d.loc[REFERENCE, 'marge_perdue'] - 1, signe=True)})")

# ---------------------------------------------------------------------------
# 5. Graphique : fréquence mensuelle des retours, par canal, sur 36 mois
# ---------------------------------------------------------------------------
par_mois = ventes.groupby(["annee_mois", "type"]).a_retour.mean().unstack("type")[ORDRE_TYPES]
fig, ax = plt.subplots(figsize=(11.5, 4.4))
positions = np.arange(len(par_mois))
for t in ORDRE_TYPES:
    ax.plot(positions, par_mois[t], color=COULEURS_TYPES[t], linewidth=2.2, marker="o", markersize=3, label=t)
ax.axhline(FREQUENCE_GLOBALE, color=viz.GRIS, linewidth=1, linestyle=":")
ax.set_ylim(0, par_mois.max().max() * 1.3)
ax.set_xticks(positions[::3], [str(m) for m in par_mois.index[::3]], rotation=45, ha="right")
ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
viz.format_pourcent(ax, axe="y")
ax.legend(frameon=False, loc="lower right", ncol=3)
viz.titre(ax, "Une ligne sur sept a un retour, chaque mois des trois exercices",
          "Part des lignes de vente avec au moins une unité retournée, par canal · pointillé : moyenne des trois exercices")
print("Figure enregistrée :", viz.enregistrer(fig, "08_retours_par_mois"))
plt.show()
ecarts_mensuels = par_mois.agg(["min", "max"]).T
print("Fréquence mensuelle la plus basse et la plus haute, par canal :")
display(ecarts_mensuels.style.format("{:.1%}"))

**Lecture**

**Un retour est fréquent mais petit.** 85,5 % des lignes n'ont aucun retour, 14,1 % ont une unité retournée, 0,4 % en ont deux, et seules 87 lignes sur 715 291 en ont trois ou quatre. Quand il y a retour, il porte sur **une seule unité dans 97,3 % des cas**, alors qu'une ligne compte en moyenne 14 unités.

**Sur les trois exercices, la fréquence ne bouge pas ; le coût baisse avec les ventes.**

| | 2019 | 2020 | 2021 | 2019 → 2021 |
|---|---|---|---|---|
| Lignes avec retour (fréquence) | 14,41 % | 14,70 % | 14,47 % | +0,06 point |
| Taux en unités | 1,39 % | 1,02 % | 0,70 % | −0,69 point |
| Taux en valeur | 1,47 % | 1,32 % | 1,32 % | −0,16 point |
| Montant des retours | 14,2 M$ | 10,2 M$ | 9,2 M$ | −35,2 % |
| Marge perdue | 8,0 M$ | 5,8 M$ | 5,3 M$ | −33,4 % |
| Chiffre d'affaires brut | 959,6 M$ | 769,2 M$ | 697,4 M$ | −27,3 % |

- **Le taux en unités est divisé par deux** (1,39 % → 0,70 %), ce qui ressemble à une forte amélioration. Mais la fréquence ne change pas : la baisse vient seulement de lignes deux fois plus grosses (10,7 puis 21,3 unités). La section 3 traite ce piège.
- **La marge perdue baisse un peu plus vite que les ventes** (−33,4 % contre −27,3 %) : la section 9 en donne la raison.

**Par canal, deux niveaux de fréquence, et une trajectoire propre aux magasins.**

| | Fréquence 2019 → 2021 | Taux en valeur 2019 → 2021 | Marge perdue 2019 | 2020 | 2021 | 2019 → 2021 |
|---|---|---|---|---|---|---|
| En ligne | 17,8 % → 16,7 % | 1,40 % → 1,28 % | 2,03 M$ | 2,00 M$ | 2,03 M$ | +0,2 % |
| Revendeur | 16,7 % → 15,1 % | 1,41 % → 1,31 % | 1,47 M$ | 1,35 M$ | 1,21 M$ | −17,9 % |
| Magasins | 12,6 % → 12,4 % | 1,54 % → 1,35 % | 4,52 M$ | 2,44 M$ | 2,10 M$ | −53,6 % |

- Les magasins ont **la fréquence la plus basse** des trois canaux, chaque exercice, mais le **taux en valeur le plus haut** : leurs lignes sont plus petites (17,1 unités en 2021 contre 25,8 en ligne), si bien qu'une unité retournée y pèse plus lourd.
- La marge perdue des magasins suit leurs ventes (−53,6 % pour un chiffre d'affaires de −48,6 %, `06`). En 2021, les retours coûtent autant en magasin (2,10 M$) qu'en ligne (2,03 M$).
- **Aucune saison** : la fréquence mensuelle reste entre 15,9 % et 18,5 % en ligne, 14,2 % et 17,2 % chez le revendeur, 11,3 % et 13,6 % en magasin, sur les 36 mois.

## 3. Le choix de la mesure

Trois taux de retour sont possibles, et ils ne racontent pas la même chose. Avant de comparer des canaux, des gammes ou des exercices, il faut choisir celui qui se compare **sans biais**.

Le portrait montre qu'un retour porte presque toujours sur **une seule unité**, quelle que soit la taille de la ligne. Une ligne de 40 unités avec un retour a donc un taux en unités de 2,5 %, une ligne de 8 unités de 12,5 %, pour le même événement. Or la taille des lignes a doublé entre 2019 et 2021 (`06_q1_facteurs_rentabilite` : unités par ligne +99,4 %). On vérifie l'effet de la taille sur chaque mesure, canal par canal.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Les trois mesures selon la taille de la ligne, par canal
# ---------------------------------------------------------------------------
BORNES_TAILLE = [0, 5, 8, 10, 12, 20, 50, np.inf]
ETIQUETTES_TAILLE = ["1 à 5", "6 à 8", "9 à 10", "11 à 12", "13 à 20", "21 à 50", "plus de 50"]
ventes["taille_ligne"] = pd.cut(ventes.quantite_vente, BORNES_TAILLE, labels=ETIQUETTES_TAILLE)
par_taille = retours_par("taille_ligne")
print("Toutes boutiques, selon le nombre d'unités vendues sur la ligne :")
montrer(par_taille)
taille_canal = retours_par(["type", "taille_ligne"])
frequence_taille = taille_canal.frequence.unstack("type")[ORDRE_TYPES]
lignes_taille = taille_canal.lignes.unstack("type")[ORDRE_TYPES]
print("Fréquence des retours selon la taille de la ligne, par canal :")
display(frequence_taille.style.format("{:.1%}", na_rep=""))
print("Répartition des lignes de chaque canal par taille :")
display((lignes_taille / lignes_taille.sum()).style.format("{:.1%}", na_rep=""))
tableaux["3_par_taille"] = par_taille
tableaux["3_frequence_taille_canal"] = frequence_taille

# Part des lignes de plus de 12 unités, par exercice et par canal : c'est là que la fréquence monte en ligne
part_grosses = ventes.assign(grosse=ventes.quantite_vente > 12).groupby(["type", "annee"]).grosse.mean().unstack("annee").loc[ORDRE_TYPES]
print("Part des lignes de plus de 12 unités, par canal et par exercice :")
display(part_grosses.style.format("{:.1%}"))
tableaux["3_part_grosses_lignes"] = part_grosses

# ---------------------------------------------------------------------------
# 2. Tests : la présence d'un retour dépend-elle de la taille de la ligne ? (par canal)
# ---------------------------------------------------------------------------
for t in ORDRE_TYPES:
    d = ventes[ventes.type == t]
    rho, p = stats.spearmanr(d.quantite_vente, d.a_retour)
    noter_test(registre, f"Mesure : la présence d'un retour dépend-elle de la taille de la ligne ({t.lower()}) ?",
               "Même fréquence de retour quelle que soit la taille de la ligne", "Spearman", len(d), f"ρ = {dec(rho)}", p)

# ---------------------------------------------------------------------------
# 3. Graphique : taux en unités et fréquence selon la taille, par canal
# ---------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
x = np.arange(len(ETIQUETTES_TAILLE))
ax1.bar(x, par_taille.taux_unites, color=viz.ALERTE)
ax1.set_xticks(x, ETIQUETTES_TAILLE, rotation=45, ha="right")
viz.format_pourcent(ax1, axe="y", decimales=1)
ax1.set_xlabel("Unités vendues sur la ligne")
viz.titre(ax1, "Le taux en unités fond avec la taille", "Unités retournées ÷ unités vendues, toutes boutiques")
for t in ORDRE_TYPES:
    ax2.plot(x, frequence_taille[t], color=COULEURS_TYPES[t], linewidth=2.2, marker="o", label=t)
ax2.set_xticks(x, ETIQUETTES_TAILLE, rotation=45, ha="right")
ax2.set_ylim(0, frequence_taille.max().max() * 1.2)
ax2.yaxis.set_major_locator(plt.MultipleLocator(0.05))
viz.format_pourcent(ax2, axe="y")
ax2.set_xlabel("Unités vendues sur la ligne")
ax2.legend(frameon=False, loc="lower right")
viz.titre(ax2, "La fréquence ne monte qu'en ligne et chez le revendeur", "Lignes avec retour ÷ lignes, par canal")
fig.tight_layout()
print("Figure enregistrée :", viz.enregistrer(fig, "08_choix_de_la_mesure"))
plt.show()

**Lecture**

- **Le taux en unités mesure surtout la taille des lignes** : 3,02 % pour les lignes de 1 à 5 unités, 0,13 % au-delà de 50, alors que la fréquence, elle, ne baisse pas. Comparer deux exercices ou deux canaux avec ce taux revient à comparer la taille de leurs lignes.
- **En magasin, la fréquence ne dépend pas de la taille de la ligne** : 12,1 % à 13,2 % dans toutes les tranches (ρ = +0,004, H0 conservée après correction).
- **En ligne et chez le revendeur, elle monte nettement au-delà de 12 unités** : de 12,8 % (1 à 5 unités) à 24,7 % (21 à 50 unités) en ligne, de 12,5 % à 22,9 % chez le revendeur (ρ = +0,110 et +0,089).
- **L'écart entre canaux est presque entièrement dans les grosses lignes** : sur les lignes de 1 à 5 unités, les trois canaux sont entre 12,4 % et 12,8 %. Or les lignes de plus de 12 unités font 25,2 %, 35,9 % puis 57,3 % des lignes en ligne de 2019 à 2021, contre 6,1 %, 6,6 % puis 33,9 % en magasin.
- Malgré des lignes bien plus grosses, la fréquence en ligne baisse de 17,8 % à 16,7 % entre 2019 et 2021 : à taille égale, les retours en ligne sont donc devenus un peu plus rares.

*Interprétation métier (hypothèse)* : une ligne est le cumul d'une journée. En ligne et chez le revendeur, une grosse ligne réunit sans doute plusieurs commandes, et chaque commande peut donner lieu à un retour ; en magasin, le volume du jour ne change pas la probabilité qu'un client rapporte un article. Nos données ne montrent ni les commandes ni les tickets : le point reste *ouvert*.

**Mesures retenues pour la suite**

| Question | Mesure | Pourquoi |
|---|---|---|
| Qui retourne le plus souvent ? | **Fréquence** (part des lignes avec retour) | La moins sensible à la taille des lignes, et pas du tout en magasin |
| Qu'est-ce que cela coûte ? | **Taux en valeur** et **marge perdue** | Ce sont des dollars, comparables partout |
| Taux en unités | Affiché pour information, **jamais utilisé pour comparer** | Biaisé par la taille des lignes, qui a doublé en deux exercices |

La taille de la ligne devient un **facteur de confusion** à contrôler pour Q4 (section 8), canal par canal.

## 4. Q3 : le poids de chaque facteur

Pour chaque facteur, on calcule la fréquence des retours par groupe, puis on teste si elle diffère d'un groupe à l'autre.

- **H0** : la fréquence des retours est la même dans tous les groupes du facteur.
- **Test** : khi-deux d'indépendance, avec le **V de Cramér** comme taille d'effet (0,1 petit, 0,3 moyen, 0,5 grand).

Le magasin et le pays sont testés sur les **magasins ouverts toute l'année** seulement : la boutique en ligne et le revendeur servent toute l'Europe. Le même calcul est refait **par exercice** pour vérifier que le classement ne change pas.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Les facteurs, sur les trois exercices réunis
# ---------------------------------------------------------------------------
FACTEURS = {
    "Type de boutique": "type", "Catégorie": "categorie", "Sous-catégorie": "sous_categorie", "Gamme": "gamme",
    "Marque": "marque", "Produit": "id_produit", "Âge du produit": "age_produit", "Ancienneté du produit": "anciennete",
    "Promotion (oui ou non)": "a_promotion", "Remise sur la ligne (oui ou non)": "a_remise", "Taille de la ligne": "taille_ligne",
    "Année": "annee", "Mois": "mois", "Jour de la semaine": "nom_jour", "Jour férié": "est_ferie",
}
FACTEURS_MAGASINS = {"Magasin": "id_boutique", "Pays (magasins)": "pays"}


def poids_facteurs(df, df_magasins):
    """V de Cramér et valeur p de chaque facteur, avec la fréquence la plus basse et la plus haute."""
    lignes = []
    for nom, colonne in list(FACTEURS.items()) + list(FACTEURS_MAGASINS.items()):
        d = df_magasins if nom in FACTEURS_MAGASINS else df
        if d[colonne].nunique() < 2:
            continue
        v_cramer, p = khi2(d.a_retour, d[colonne])
        parts = d.groupby(colonne, observed=True).a_retour.mean()
        lignes.append({"facteur": nom, "groupes": d[colonne].nunique(), "fréquence la plus basse": parts.min(),
                       "fréquence la plus haute": parts.max(), "V de Cramér": v_cramer, "p": p, "n": len(d)})
    return pd.DataFrame(lignes).set_index("facteur")


poids = poids_facteurs(ventes, magasins_lignes).sort_values("V de Cramér", ascending=False)
for nom, ligne in poids.iterrows():
    noter_test(registre, f"Q3 : la fréquence des retours dépend-elle du facteur « {nom} » ?",
               "Même fréquence de retour dans tous les groupes", "Khi-deux", int(ligne.n), f"V = {dec(ligne['V de Cramér'], signe=False)}", ligne.p)
FORMAT_P = lambda p: "< 1e-300" if p == 0 else f"{p:.2g}"
print("Poids de chaque facteur sur la fréquence des retours, trois exercices réunis :")
display(poids.drop(columns="n").style.format({"fréquence la plus basse": "{:.1%}", "fréquence la plus haute": "{:.1%}",
                                              "V de Cramér": "{:.3f}", "p": FORMAT_P}))
tableaux["4_poids_des_facteurs"] = poids

# ---------------------------------------------------------------------------
# 2. Vérification par exercice : le V de Cramér de chaque facteur, année par année
# ---------------------------------------------------------------------------
v_par_annee = pd.DataFrame({a: poids_facteurs(ventes[ventes.annee == a], magasins_lignes[magasins_lignes.annee == a])["V de Cramér"]
                            for a in ANNEES}).reindex(poids.index).drop(index="Année")
print("V de Cramér de chaque facteur, par exercice :")
display(v_par_annee.style.format("{:.3f}", na_rep=""))
tableaux["4_poids_par_exercice"] = v_par_annee

# Détail des facteurs les plus lisibles
for nom in ["Catégorie", "Gamme", "Marque", "Pays (magasins)", "Mois", "Jour de la semaine"]:
    colonne = {**FACTEURS, **FACTEURS_MAGASINS}[nom]
    table = retours_par(colonne, magasins_lignes if nom in FACTEURS_MAGASINS else None).sort_values("frequence", ascending=False)
    tableaux[f"4_{colonne}"] = table
print("Retours par catégorie, trois exercices :")
montrer(tableaux["4_categorie"])

# ---------------------------------------------------------------------------
# 3. Sévérité COVID-19 : 2020 et 2021 seulement, magasin × mois, à mois égal
# ---------------------------------------------------------------------------
covid = (magasins_lignes[magasins_lignes.annee.isin(config.ANNEES_ATTENDUES)]
         .groupby(["id_boutique", "annee", "mois"]).agg(frequence=("a_retour", "mean"), severite=("severite_covid", "mean"),
                                                        lignes=("id_vente", "size")).reset_index())
for col in ["frequence", "severite"]:
    covid[col + "_centre"] = covid[col] - covid.groupby(["annee", "mois"])[col].transform("mean")
rho_covid, p_covid = stats.spearmanr(covid.severite_centre, covid.frequence_centre)
noter_test(registre, "Q3 : la sévérité locale du COVID-19 va-t-elle avec la fréquence des retours d'un magasin (à mois égal, 2020-2021) ?",
           "Aucun lien entre sévérité et fréquence des retours", "Spearman", len(covid), f"ρ = {dec(rho_covid)}", p_covid)

# ---------------------------------------------------------------------------
# 4. Graphique : le poids de chaque facteur
# ---------------------------------------------------------------------------
trie = poids["V de Cramér"].sort_values()
fig, ax = plt.subplots(figsize=(9.5, 5.6))
ax.barh(trie.index, trie.values, color=[viz.PRINCIPALE if v >= 0.1 else viz.GRIS for v in trie.values])
for y, v in enumerate(trie.values):
    ax.text(v + 0.002, y, f"{v:.3f}".replace(".", ","), va="center", fontsize=9, color=viz.FONCE)
ax.axvline(0.1, color=viz.ALERTE, linewidth=1.5, linestyle="--")
ax.text(0.098, -0.9, "seuil d'un petit effet (0,1)", color=viz.ALERTE, fontsize=9, va="center", ha="right")
ax.set_xlim(0, 0.12)
ax.xaxis.set_major_formatter(lambda valeur, _: f"{valeur:.2f}".replace(".", ","))
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
viz.titre(ax, "Aucun facteur n'atteint un petit effet sur la fréquence des retours",
          "V de Cramér entre chaque facteur et la présence d'un retour sur la ligne · trois exercices")
print("Figure enregistrée :", viz.enregistrer(fig, "08_poids_des_facteurs"))
plt.show()

**Lecture**

**Aucun facteur n'atteint le seuil d'un petit effet (V de 0,1) sur les trois exercices réunis.** C'est le résultat principal de Q3 : la fréquence des retours est presque la même partout.

| Facteur | V de Cramér | Fréquence, du groupe le plus bas au plus haut | Lecture |
|---|---|---|---|
| Taille de la ligne | 0,085 | 12,5 % à 21,7 % | le seul facteur qui approche un petit effet (0,121 en 2020) ; il joue en ligne et chez le revendeur (section 3) |
| Produit | 0,061 | 2,6 % à 32,0 % | H0 conservée après correction (p = 0,032) : les écarts sont ceux du hasard sur 2 516 produits (section 6) |
| Type de boutique | 0,059 | 12,6 % à 17,2 % | magasins plus bas, surtout par la taille des lignes |
| Magasin | 0,020 | 9,0 % à 13,9 % | écarts réels mais non durables (section 6) |
| Sous-catégorie, remise, marque, catégorie, pays | 0,010 à 0,013 | un à deux points d'écart | significatifs, sans portée pratique |
| Mois, jour férié, jour de la semaine, année | 0,004 à 0,006 | moins d'un point | aucune saison |
| Promotion, ancienneté, gamme, âge du produit | 0,001 à 0,002 | 14,4 % à 14,6 % | H0 conservée : aucun lien |

- **Par exercice, le classement ne change pas.** Trois valeurs dépassent 0,1, et toutes s'expliquent : la taille de la ligne en 2020 (0,121), et le produit en 2020 (0,105) et en 2021 (0,124), parce que sur un seul exercice chaque produit compte moins de lignes (168 534 lignes pour 2 504 produits en 2021), ce qui gonfle le V par simple bruit.
- La catégorie *Games and Toys* a la fréquence la plus haute (15,8 %) avec des lignes parmi les plus grosses (26 unités, contre 10 à 12 pour six catégories sur huit) : c'est sans doute encore l'effet de taille, pas un défaut de la catégorie.
- **La pandémie ne joue pas** : à mois égal, sur 2020 et 2021, la sévérité locale du COVID-19 n'a aucun lien avec la fréquence des retours d'un magasin (ρ = −0,003 sur 1 248 couples magasin × mois, p = 0,9).

**En clair** : il n'y a pas de « pattern » de retour au sens où un produit, une catégorie, une marque, une boutique ou une période concentrerait les retours. Aucune cause identifiable dans nos données, en dehors de la taille des lignes en ligne et chez le revendeur.

## 5. Q3 : l'âge du produit, la gamme et le canal

`06b_q1_chute_des_ventes` a établi trois mouvements : l'assortiment **vieillit** (552, 429 puis 272 lancements), les clients **descendent en gamme** (l'économique passe de 10,6 % à 21,6 % du chiffre d'affaires) et le **canal en ligne** prend le pas sur les magasins. Si l'un de ces mouvements s'accompagnait de plus de retours, il aggraverait la perte. On vérifie les trois, exercice par exercice.

- **H0 (âge)** : un produit ancien n'est pas plus souvent retourné qu'une nouveauté de l'année. *Interprétation métier (hypothèse)* à tester : un produit en fin de vie, moins à jour, décevrait davantage.
- **H0 (gamme)** : l'entrée de gamme n'est pas plus souvent retournée que le standard et le luxe. *Hypothèse* : un produit bon marché, de moindre qualité, serait rapporté plus souvent.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Âge du produit : par exercice et par canal
# ---------------------------------------------------------------------------
age_annee = retours_par(["age_produit", "annee"]).loc[ORDRE_AGES]
print("Retours selon l'âge du produit, par exercice :")
montrer(age_annee)
age_canal = ventes.groupby(["type", "age_produit"]).a_retour.mean().unstack("age_produit").loc[ORDRE_TYPES, ORDRE_AGES]
age_canal["écart (ancien − nouveauté)"] = age_canal[ORDRE_AGES[1]] - age_canal[ORDRE_AGES[0]]
print("Fréquence des retours selon l'âge du produit, par canal, trois exercices :")
display(age_canal.style.format("{:.2%}"))
anciennete = retours_par(["anciennete"])
print("Selon l'ancienneté du produit (années écoulées depuis sa mise en vente) :")
montrer(anciennete)
tableaux["5_age_par_exercice"] = age_annee
tableaux["5_age_par_canal"] = age_canal
tableaux["5_anciennete"] = anciennete

# Test par exercice (le pool mélange des années où la part des nouveautés n'est pas la même)
for a in ANNEES:
    d = ventes[ventes.annee == a]
    v_age, p_age = khi2(d.a_retour, d.age_produit)
    ecart = d[d.age_produit == ORDRE_AGES[1]].a_retour.mean() - d[d.age_produit == ORDRE_AGES[0]].a_retour.mean()
    noter_test(registre, f"Q3 : un produit ancien est-il plus souvent retourné qu'une nouveauté ({a}) ?",
               "Même fréquence pour les nouveautés et les produits anciens", "Khi-deux", len(d), f"V = {dec(v_age, signe=False)} · écart {points(ecart, 2)}", p_age)

# ---------------------------------------------------------------------------
# 2. Gamme : par exercice et par canal
# ---------------------------------------------------------------------------
gamme_annee = retours_par(["gamme", "annee"]).loc[ORDRE_GAMMES]
print("Retours selon la gamme, par exercice :")
montrer(gamme_annee)
gamme_canal = ventes[ventes.annee == EXERCICE].groupby(["type", "gamme"]).a_retour.mean().unstack("gamme").loc[ORDRE_TYPES, ORDRE_GAMMES]
print(f"Fréquence des retours selon la gamme, par canal, {EXERCICE} :")
display(gamme_canal.style.format("{:.2%}"))
tableaux["5_gamme_par_exercice"] = gamme_annee
tableaux["5_gamme_par_canal"] = gamme_canal
for a in ANNEES:
    d = ventes[ventes.annee == a]
    v_g, p_g = khi2(d.a_retour, d.gamme)
    noter_test(registre, f"Q3 : la fréquence des retours dépend-elle de la gamme ({a}) ?",
               "Même fréquence dans les trois gammes", "Khi-deux", len(d), f"V = {dec(v_g, signe=False)}", p_g)

# ---------------------------------------------------------------------------
# 3. Évolution 2019 → 2021, segment par segment
# ---------------------------------------------------------------------------
segments = {"Ensemble": ventes.assign(seg="Ensemble"), "Canal": ventes.assign(seg=ventes.type),
            "Gamme": ventes.assign(seg=ventes.gamme), "Âge du produit": ventes.assign(seg=ventes.age_produit)}
lignes_evol = []
for famille, d in segments.items():
    t = retours_par(["seg", "annee"], d)
    for seg in t.index.get_level_values("seg").unique():
        s = t.loc[seg]
        lignes_evol.append({"famille": famille, "segment": seg,
                            **{f"fréquence {a}": s.loc[a, "frequence"] for a in ANNEES},
                            f"écart {REFERENCE} → {EXERCICE}": s.loc[EXERCICE, "frequence"] - s.loc[REFERENCE, "frequence"],
                            f"taux en valeur {REFERENCE}": s.loc[REFERENCE, "taux_valeur"], f"taux en valeur {EXERCICE}": s.loc[EXERCICE, "taux_valeur"],
                            f"marge perdue {REFERENCE}": s.loc[REFERENCE, "marge_perdue"], f"marge perdue {EXERCICE}": s.loc[EXERCICE, "marge_perdue"],
                            f"évolution de la marge perdue": s.loc[EXERCICE, "marge_perdue"] / s.loc[REFERENCE, "marge_perdue"] - 1})
evol_segments = pd.DataFrame(lignes_evol).set_index(["famille", "segment"])
print(f"Évolution des retours de {REFERENCE} à {EXERCICE}, par segment :")
display(evol_segments.style.format({c: ("{:.1%}" if c.startswith("fréquence") else "{:+.2%}" if c.startswith("écart") else "{:.2%}" if c.startswith("taux")
                                        else "{:+.1%}" if c.startswith("évolution") else "{:,.0f}") for c in evol_segments.columns}))
tableaux["5_evolution_par_segment"] = evol_segments

# Part des lignes avec retour qui viennent des produits anciens, de l'entrée de gamme et des magasins, par exercice
parts_retours = pd.DataFrame({
    "produits anciens": ventes[ventes.a_retour].groupby("annee").age_produit.apply(lambda s: (s == ORDRE_AGES[1]).mean()),
    "gamme économique": ventes[ventes.a_retour].groupby("annee").gamme.apply(lambda s: (s == "Économique").mean()),
    "magasins": ventes[ventes.a_retour].groupby("annee").type.apply(lambda s: (s == "Magasins").mean()),
})
parts_lignes = pd.DataFrame({
    "produits anciens": ventes.groupby("annee").age_produit.apply(lambda s: (s == ORDRE_AGES[1]).mean()),
    "gamme économique": ventes.groupby("annee").gamme.apply(lambda s: (s == "Économique").mean()),
    "magasins": ventes.groupby("annee").type.apply(lambda s: (s == "Magasins").mean()),
})
print("Part des lignes avec retour / part de toutes les lignes, par exercice :")
display(pd.concat({"lignes avec retour": parts_retours, "toutes les lignes": parts_lignes}, axis=1).style.format("{:.1%}"))

# ---------------------------------------------------------------------------
# 4. Graphique : fréquence par âge et par gamme, sur les trois exercices
# ---------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
largeur = 0.26
for i, a in enumerate(ANNEES):
    f_age = age_annee.frequence.xs(a, level="annee").reindex(ORDRE_AGES)
    ax1.bar(np.arange(2) + (i - 1) * largeur, f_age, width=largeur, color=COULEURS_ANNEES[a], label=str(a))
    f_gamme = gamme_annee.frequence.xs(a, level="annee").reindex(ORDRE_GAMMES)
    ax2.bar(np.arange(3) + (i - 1) * largeur, f_gamme, width=largeur, color=COULEURS_ANNEES[a], label=str(a))
ax1.set_xticks(range(2), ORDRE_AGES)
ax2.set_xticks(range(3), ORDRE_GAMMES)
for ax in (ax1, ax2):
    ax.axhline(FREQUENCE_GLOBALE, color=viz.ALERTE, linewidth=1, linestyle="--")
    ax.set_ylim(0, 0.2)
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
    viz.format_pourcent(ax, axe="y")
ax1.legend(frameon=False, loc="upper center", ncol=3)
viz.titre(ax1, "Un produit ancien n'est pas plus retourné", "Part des lignes avec retour, selon l'âge du produit")
viz.titre(ax2, "L'entrée de gamme non plus", "Part des lignes avec retour, selon la gamme")
fig.tight_layout()
print("Figure enregistrée :", viz.enregistrer(fig, "08_retours_age_gamme"))
plt.show()

**Lecture**

**L'âge du produit ne change rien aux retours.**

| Fréquence | 2019 | 2020 | 2021 | Écart ancien − nouveauté |
|---|---|---|---|---|
| Nouveauté de l'année | 14,34 % | 14,70 % | 14,29 % | |
| Produit plus ancien | 14,44 % | 14,70 % | 14,49 % | +0,10 · −0,00 · +0,20 point |

- H0 est conservée les trois exercices (p de 0,49 à 0,99), dans les trois canaux (écarts de −0,07 à −0,11 point) et quelle que soit l'ancienneté (14,42 % pour une nouveauté, 14,56 % à deux ans, 14,55 % à trois ans et plus).
- Le vieillissement de l'assortiment (`06b`) ne s'accompagne donc d'**aucun surcroît de retours** : les produits anciens font 91,4 % des lignes avec retour en 2021, exactement leur part des lignes (91,3 %).
- La marge perdue sur les nouveautés tombe de 3,03 M$ à 0,77 M$ (−74,6 %) uniquement parce qu'il y a moins de nouveautés.

**L'entrée de gamme n'est pas plus souvent retournée.**

| Fréquence | 2019 | 2020 | 2021 | Taux en valeur 2019 → 2021 | Marge perdue 2019 → 2021 |
|---|---|---|---|---|---|
| Économique | 14,39 % | 14,90 % | 14,53 % | 2,51 % → 0,63 % | 1,29 → 0,47 M$ (−63,3 %) |
| Standard | 14,38 % | 14,64 % | 14,49 % | 1,33 % → 1,46 % | 4,46 → 2,93 M$ (−34,4 %) |
| Luxe | 14,66 % | 14,32 % | 14,25 % | 1,41 % → 1,58 % | 2,27 → 1,94 M$ (−14,6 %) |

- H0 est conservée chaque exercice (p de 0,063 à 0,58) ; en 2021, dans chaque canal, les trois gammes sont à moins de 0,8 point l'une de l'autre.
- **La descente en gamme fait même baisser le coût des retours** : les lignes de la gamme économique sont passées de 8,7 à 39,4 unités, et un retour d'une unité sur une ligne de 39 produits à 66 $ ne coûte presque rien (taux en valeur de 0,63 %).
- À l'inverse, le taux en valeur du luxe monte (1,41 % → 1,58 %) parce que ses lignes rétrécissent un peu (13,0 → 11,5 unités) : à fréquence égale, chaque retour y pèse davantage.

**Le canal.** Le déplacement des ventes vers la boutique en ligne porte les ventes vers le canal où les retours sont les plus fréquents (16,7 % contre 12,4 % en 2021), mais pas les plus coûteux : son taux en valeur (1,28 %) est inférieur à celui des magasins (1,35 %). Les magasins font 37,1 % des lignes avec retour en 2021 pour 43,2 % des lignes.

**Verdict** : aucun des trois mouvements établis par `06b` (vieillissement, descente en gamme, glissement vers la boutique en ligne) n'augmente le coût des retours.

## 6. Q3 : y a-t-il des produits ou des magasins anormaux ?

**Le problème.** Un produit vendu sur 50 lignes peut afficher 25 % de retours par pur hasard. Classer les produits par fréquence de retour ferait donc ressortir les **petits** produits, pas les produits à problème.

**La standardisation.** On compare chaque produit à ce que le hasard seul produirait. Si tous les produits avaient la même fréquence *p*, la fréquence observée sur *n* lignes varierait autour de *p* avec un écart-type de √(p(1 − p) ÷ n). L'**écart standardisé** (score z) mesure à combien d'écarts-types le produit se trouve de la moyenne :

> z = (fréquence observée − p) ÷ √(p(1 − p) ÷ n)

Si le hasard explique tout, les scores z ont un écart-type de 1, et moins de 3 sur 1 000 dépassent 3 en valeur absolue.

Pour les magasins, on travaille **par exercice**, sur les magasins ouverts toute l'année, avec comme référence la fréquence moyenne des magasins de l'exercice. Un magasin réellement « à problème » devrait l'être **tous les ans** : on mesure donc aussi si les scores d'un magasin se répètent d'un exercice à l'autre.

In [ ]:
def standardiser(df, cle, reference):
    """Écart standardisé (score z) de la fréquence des retours de chaque groupe par rapport à une référence."""
    g = df.groupby(cle, observed=True).a_retour.agg(lignes="size", frequence="mean")
    g["attendu"] = reference
    g["z"] = (g.frequence - reference) / np.sqrt(reference * (1 - reference) / g.lignes)
    return g


SEUIL_Z = 3
resume_z = []

# ---------------------------------------------------------------------------
# 1. Produits, trois exercices réunis
# ---------------------------------------------------------------------------
z_produits = standardiser(ventes, "id_produit", FREQUENCE_GLOBALE)
z_produits = z_produits.join(ventes.groupby("id_produit")[["nom_produit", "categorie", "gamme"]].first())
extremes_p = z_produits[z_produits.z.abs() > SEUIL_Z]
attendus_p = 2 * stats.norm.sf(SEUIL_Z) * len(z_produits)
resume_z.append({"population": "Produits, trois exercices", "groupes": len(z_produits), "écart-type des z": z_produits.z.std(),
                 "cas au-delà de 3": len(extremes_p), "attendus par hasard": attendus_p})
print(f"Produits au-delà de {SEUIL_Z} écarts-types : {len(extremes_p)} sur {nombre(len(z_produits))} (le hasard seul en donnerait environ {attendus_p:.0f})")
display(extremes_p.sort_values("z", ascending=False)[["nom_produit", "categorie", "gamme", "lignes", "frequence", "z"]]
        .style.format({"lignes": "{:,.0f}", "frequence": "{:.1%}", "z": "{:+.1f}"}))
tableaux["6_z_produits"] = z_produits.sort_values("z", ascending=False)

# ---------------------------------------------------------------------------
# 2. Magasins ouverts toute l'année, par exercice
# ---------------------------------------------------------------------------
z_magasins = {}
for a in ANNEES:
    d = magasins_lignes[magasins_lignes.annee == a]
    z = standardiser(d, "nom_boutique", d.a_retour.mean())
    z_magasins[a] = z
    extremes = z[z.z.abs() > SEUIL_Z]
    resume_z.append({"population": f"Magasins, {a}", "groupes": len(z), "écart-type des z": z.z.std(),
                     "cas au-delà de 3": len(extremes), "attendus par hasard": 2 * stats.norm.sf(SEUIL_Z) * len(z)})
z_mag = pd.concat({a: z.z for a, z in z_magasins.items()}, axis=1)
z_mag["fréquence moyenne"] = magasins_lignes.groupby("nom_boutique").a_retour.mean()
resume_z = pd.DataFrame(resume_z).set_index("population")
display(resume_z.style.format({"groupes": "{:,.0f}", "écart-type des z": "{:.2f}", "cas au-delà de 3": "{:.0f}", "attendus par hasard": "{:.1f}"}))
tableaux["6_resume_standardisation"] = resume_z

# Un magasin au-dessus de la moyenne une année l'est-il encore l'année suivante ?
stabilite = {f"{a} et {b}": z_mag[a].corr(z_mag[b], method="spearman") for a, b in [(REFERENCE, BASCULE), (BASCULE, EXERCICE), (REFERENCE, EXERCICE)]}
print("Corrélation de rang des scores z d'un magasin d'un exercice à l'autre :", " · ".join(f"{k} : {dec(v, 2)}" for k, v in stabilite.items()))
communs = z_mag.dropna(subset=[REFERENCE, EXERCICE])
rho_stab, p_stab = stats.spearmanr(communs[REFERENCE], communs[EXERCICE])
noter_test(registre, f"Q3 : un magasin aux retours fréquents en {REFERENCE} l'est-il encore en {EXERCICE} ?",
           "Aucun lien entre les scores z d'un magasin d'un exercice à l'autre", "Spearman", len(communs), f"ρ = {dec(rho_stab, 2)}", p_stab)
toujours_hauts = z_mag[(z_mag[ANNEES] > 2).all(axis=1)].index.tolist()
toujours_bas = z_mag[(z_mag[ANNEES] < -2).all(axis=1)].index.tolist()
print("Magasins au-delà de +2 écarts-types les trois exercices :", toujours_hauts or "aucun",
      "· en deçà de −2 les trois exercices :", toujours_bas or "aucun")
print("Les cinq magasins aux retours les plus rares et les cinq aux retours les plus fréquents (trois exercices) :")
classement = z_mag.sort_values("fréquence moyenne")
display(pd.concat([classement.head(5), classement.tail(5)]).style.format({a: "{:+.1f}" for a in ANNEES} | {"fréquence moyenne": "{:.1%}"}, na_rep="(non ouvert)"))
tableaux["6_z_magasins"] = classement

# ---------------------------------------------------------------------------
# 3. Graphique en entonnoir : chaque produit selon son nombre de lignes et sa fréquence de retour
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4.8))
normaux = z_produits[z_produits.z.abs() <= SEUIL_Z]
ax.scatter(normaux.lignes, normaux.frequence, s=6, color=viz.PRINCIPALE, alpha=0.35, linewidths=0)
ax.scatter(extremes_p.lignes, extremes_p.frequence, s=22, color=viz.ALERTE, linewidths=0, label=f"Au-delà de {SEUIL_Z} écarts-types")
n = np.linspace(z_produits.lignes.min(), z_produits.lignes.max(), 300)
marge_hasard = SEUIL_Z * np.sqrt(FREQUENCE_GLOBALE * (1 - FREQUENCE_GLOBALE) / n)
ax.plot(n, FREQUENCE_GLOBALE + marge_hasard, color=viz.ACCENT, linewidth=1.8, label="Limites du hasard (3 écarts-types)")
ax.plot(n, np.clip(FREQUENCE_GLOBALE - marge_hasard, 0, None), color=viz.ACCENT, linewidth=1.8)
ax.axhline(FREQUENCE_GLOBALE, color=viz.GRIS, linewidth=1, linestyle=":")
ax.set_xlabel("Nombre de lignes de vente du produit, trois exercices")
viz.format_pourcent(ax, axe="y")
ax.legend(frameon=False, loc="upper right")
viz.titre(ax, "Presque tous les produits restent dans les limites du hasard",
          "Un point par produit · part des lignes avec retour selon le nombre de lignes du produit")
print("Figure enregistrée :", viz.enregistrer(fig, "08_entonnoir_produits"))
plt.show()

**Lecture**

- **Produits : aucun produit à problème.** L'écart-type des scores z est de 1,03, presque exactement ce que donne le hasard (1). 14 produits sur 2 516 dépassent 3 écarts-types, contre 7 attendus par hasard. Les plus hauts affichent 20,5 % à 32,0 % de lignes avec retour, mais sur 46 à 336 lignes en trois exercices : ce sont des fluctuations. Sur le graphique, le nuage épouse l'entonnoir.
- **Magasins : des écarts réels une année, qui ne se répètent pas.**

| Magasins ouverts toute l'année | 2019 | 2020 | 2021 |
|---|---|---|---|
| Nombre | 48 | 51 | 53 |
| Écart-type des scores z (hasard : 1) | 1,65 | 1,97 | 1,13 |
| Magasins au-delà de 3 écarts-types (hasard : 0,1) | 4 | 8 | 1 |

Chaque exercice, la dispersion dépasse ce que donnerait le hasard. Mais un magasin aux retours fréquents une année ne l'est pas l'année suivante : la corrélation de rang des scores z est de −0,04 (2019 et 2020), +0,06 (2020 et 2021) et −0,03 (2019 et 2021, p = 0,84). Aucun magasin n'est au-delà de +2 ou en deçà de −2 écarts-types les trois années. Exemples : *Torino* (+1,7, +3,9, +0,7), *Moscow No.1* (+0,7, +4,2, −0,5), *Cambridge* (−4,2, −1,1, −0,2). La fréquence la plus basse sur trois exercices est celle de *Marseille* (9,0 %), sur sa seule année complète.

**Conséquence** : il n'y a pas de magasin où concentrer une action sur les retours. Aucune cause identifiable dans nos données pour les écarts d'une année.

**Méthode à retenir** : un classement brut des produits par fréquence de retour aurait désigné de « mauvais » produits qui ne le sont pas. La standardisation, et pour les magasins la vérification d'un exercice à l'autre, évitent cette erreur.

## 7. Q3 : retourne-t-on davantage ce qui est cher ?

On regroupe les lignes en dix tranches de prix unitaire de même effectif.

- **H0** : la fréquence des retours ne dépend pas du prix unitaire.
- **Test** : corrélation de Spearman entre le prix unitaire et la présence d'un retour, au niveau de la ligne, puis au niveau du produit.

In [ ]:
ventes["tranche_prix"] = pd.qcut(ventes.prix_unitaire, 10, duplicates="drop")
par_prix = retours_par("tranche_prix")
montrer(par_prix)
tableaux["7_par_tranche_de_prix"] = par_prix.rename(index=str)
part_haute = par_prix.marge_perdue.iloc[-1] / par_prix.marge_perdue.sum()
print(f"La tranche de prix la plus haute porte {pct(part_haute)} de la marge perdue, pour 10 % des lignes")

rho_prix, p_prix = stats.spearmanr(ventes.prix_unitaire, ventes.a_retour)
noter_test(registre, "Q3 : la fréquence des retours dépend-elle du prix unitaire (par ligne) ?",
           "Aucun lien entre le prix unitaire et la présence d'un retour", "Spearman", len(ventes), f"ρ = {dec(rho_prix)}", p_prix)
produits = ventes.groupby("id_produit").agg(prix=("prix_unitaire", "mean"), frequence=("a_retour", "mean"))
rho_produit, p_produit = stats.spearmanr(produits.prix, produits.frequence)
noter_test(registre, "Q3 : les produits chers sont-ils plus souvent retournés (par produit) ?",
           "Aucun lien entre le prix d'un produit et la fréquence de ses retours", "Spearman", len(produits), f"ρ = {dec(rho_produit)}", p_produit)

**Lecture**

- **La fréquence des retours ne dépend pas du prix** : elle reste entre 14,29 % et 14,97 % dans les dix tranches de prix. Les deux corrélations sont nulles (ρ = −0,004 par ligne, −0,046 par produit) et aucune ne passe le seuil corrigé (p = 0,0017 et 0,021, pour un seuil de 0,0014). On ne retourne pas plus souvent un produit cher.
- **Le taux en unités semble monter avec le prix** (de 0,41 % à 1,47 %) : c'est le piège de la section 3, les produits bon marché se vendant par lignes de 37 unités, les plus chers par lignes de 10.
- **Ce qui dépend du prix, c'est le coût** : la tranche la plus chère (plus de 699 $) porte à elle seule 43,4 % de la marge perdue des trois exercices (8,3 M$ sur 19,1 M$), pour 10 % des lignes. C'est ce que chiffre la section 9.

## 8. Q4 : les promotions et les remises augmentent-elles les retours ?

**La comparaison directe**, d'abord : fréquence des retours selon le taux annoncé, puis selon que la ligne a reçu une remise ou non, canal par canal.

**Les facteurs de confusion**, ensuite. La section 3 a montré qu'en ligne et chez le revendeur, une grosse ligne a plus souvent un retour. Or une grosse ligne a aussi plus de chances de contenir une unité remisée. Un lien apparent entre remise et retour peut donc venir de la seule taille des lignes. La période (année, mois), la catégorie et la gamme sont aussi contrôlées.

On utilise une **régression logistique** : elle estime l'effet de la remise sur la probabilité de retour **à taille de ligne, catégorie, gamme, mois et année identiques**. Elle est ajustée **canal par canal**, puisque le rôle de la taille n'est pas le même en magasin.

- **H0** : à caractéristiques égales, la remise ne change pas la probabilité de retour (rapport de cotes = 1).

In [ ]:
# ---------------------------------------------------------------------------
# 1. Comparaison directe
# ---------------------------------------------------------------------------
par_taux = retours_par("taux_remise_annonce")
print("Retours selon le taux annoncé de la promotion, trois exercices :")
display(par_taux.style.format({c: f for c, f in FORMATS_RETOURS.items() if c in par_taux.columns}).format_index(lambda t: pct(t, 0)))
tableaux["8_par_taux_annonce"] = par_taux

ventes["statut_remise"] = np.select([~ventes.a_promotion, ~ventes.a_remise], ["Hors promotion", "En promotion, sans remise"], "Ligne remisée")
ORDRE_STATUTS = ["Hors promotion", "En promotion, sans remise", "Ligne remisée"]
statut_canal = retours_par(["type", "statut_remise"]).loc[ORDRE_TYPES]
print("Retours selon la promotion et la remise, par canal :")
montrer(statut_canal)
tableaux["8_par_statut_remise"] = statut_canal
statut_annee = ventes.groupby(["annee", "statut_remise"]).a_retour.mean().unstack()[ORDRE_STATUTS]
print("Fréquence des retours selon la promotion et la remise, par exercice :")
display(statut_annee.style.format("{:.2%}"))

v_taux, p_taux = khi2(ventes.a_retour, ventes.taux_remise_annonce)
noter_test(registre, "Q4, direct : la fréquence des retours dépend-elle du taux annoncé ?",
           "Même fréquence de retour pour tous les taux annoncés", "Khi-deux", len(ventes), f"V = {dec(v_taux, signe=False)}", p_taux)
rho_remise, p_rho = stats.spearmanr(ventes.taux_remise_effectif, ventes.a_retour)
noter_test(registre, "Q4, direct : corrélation entre le taux de remise effectif et la présence d'un retour",
           "Aucune corrélation entre le taux de remise effectif et les retours", "Spearman", len(ventes), f"ρ = {dec(rho_remise)}", p_rho)
v_hors, p_hors = khi2(ventes.a_retour, ventes.promo_hors_periode)
noter_test(registre, "Q4, direct : les lignes de promotion hors période (janvier) ont-elles plus de retours ?",
           "Même fréquence de retour dans et hors période de promotion", "Khi-deux", len(ventes), f"V = {dec(v_hors, signe=False)}", p_hors)

# ---------------------------------------------------------------------------
# 2. Régression logistique : l'effet de la remise, à caractéristiques égales, par canal
# ---------------------------------------------------------------------------
donnees = pd.DataFrame({
    "retour": ventes.a_retour.astype(int),
    "remisee": ventes.a_remise.astype(int),                    # 1 si la ligne a au moins une unité remisée
    "log_quantite": np.log(ventes.quantite_vente),             # taille de la ligne (logarithme : effet proportionnel)
    "type": ventes.type, "categorie": ventes.categorie, "gamme": ventes.gamme,
    "mois": ventes.mois, "annee": ventes.annee,
})
CONTROLES = "log_quantite + C(categorie) + C(gamme) + C(mois) + C(annee)"


def rapport_de_cotes(modele, variable):
    """Rapport de cotes de `variable` avec son intervalle à 95 % et sa valeur p."""
    bas, haut = np.exp(modele.conf_int().loc[variable])
    return {"rapport de cotes": np.exp(modele.params[variable]), "borne basse (95 %)": bas,
            "borne haute (95 %)": haut, "p": modele.pvalues[variable]}


resultats, modeles = {}, {}
for t in ORDRE_TYPES + ["Toutes boutiques"]:
    d = donnees if t == "Toutes boutiques" else donnees[donnees.type == t]
    formule = f"retour ~ remisee + {CONTROLES}" + (" + C(type) * log_quantite" if t == "Toutes boutiques" else "")
    brut = smf.logit("retour ~ remisee", data=d).fit(disp=0)
    controle = smf.logit(formule, data=d).fit(disp=0)
    modeles[t] = (controle, d)
    resultats[(t, "remise, sans contrôle")] = rapport_de_cotes(brut, "remisee")
    resultats[(t, "remise, à caractéristiques égales")] = rapport_de_cotes(controle, "remisee")
    resultats[(t, "taille de la ligne (quantité × 2,7)")] = rapport_de_cotes(controle, "log_quantite")
    # Ce que cela représente : probabilité de retour prédite pour les mêmes lignes, sans puis avec remise
    p_sans = controle.predict(d.assign(remisee=0)).mean()
    p_avec = controle.predict(d.assign(remisee=1)).mean()
    resultats[(t, "remise, à caractéristiques égales")] |= {"probabilité sans remise": p_sans, "probabilité avec remise": p_avec}
regression = pd.DataFrame(resultats).T
print("Effet de la remise et de la taille sur la probabilité de retour (régression logistique) :")
display(regression.style.format({"rapport de cotes": "{:.3f}", "borne basse (95 %)": "{:.3f}", "borne haute (95 %)": "{:.3f}", "p": FORMAT_P,
                                 "probabilité sans remise": "{:.1%}", "probabilité avec remise": "{:.1%}"}, na_rep=""))
tableaux["8_regression_logistique"] = regression.reset_index().rename(columns={"level_0": "canal", "level_1": "variable"})
for t in ORDRE_TYPES:
    r = resultats[(t, "remise, à caractéristiques égales")]
    noter_test(registre, f"Q4, contrôlé : à caractéristiques égales, une ligne remisée a-t-elle plus souvent un retour ({t.lower()}) ?",
               "Rapport de cotes égal à 1", "Régression logistique", int((donnees.type == t).sum()),
               f"rapport de cotes = {dec(r['rapport de cotes'], signe=False)} [{dec(r['borne basse (95 %)'], signe=False)} ; {dec(r['borne haute (95 %)'], signe=False)}]", r["p"])

# ---------------------------------------------------------------------------
# 3. Les variables disponibles permettent-elles de prédire un retour ?
# ---------------------------------------------------------------------------
# L'aire sous la courbe ROC (AUC) vaut 0,5 pour un tirage au hasard et 1 pour une prédiction parfaite.
# On la calcule par le test de Mann-Whitney : AUC = (corrélation de rang + 1) ÷ 2.
qualite = {}
for t, (modele, d) in modeles.items():
    pred = modele.predict(d)
    effet_auc, _ = mann_whitney(pred[d.retour == 1], pred[d.retour == 0])
    qualite[t] = {"AUC": (effet_auc + 1) / 2, "pseudo-R² de McFadden": modele.prsquared}
qualite = pd.DataFrame(qualite).T
print("Capacité du modèle à distinguer les lignes avec retour (0,5 = hasard, 1 = parfait) :")
display(qualite.style.format("{:.3f}"))
tableaux["8_qualite_du_modele"] = qualite

# ---------------------------------------------------------------------------
# 4. Ordre de grandeur, exercice 2021 : marge perdue attribuable au surcroît de retours des lignes remisées
# ---------------------------------------------------------------------------
attribuable = {}
for t in ORDRE_TYPES:
    modele, d = modeles[t]
    d_ex = d[(d.annee == EXERCICE) & (d.remisee == 1)]
    retours_en_plus = (modele.predict(d_ex) - modele.predict(d_ex.assign(remisee=0))).sum()
    lignes_ex = ventes[(ventes.annee == EXERCICE) & (ventes.type == t) & ventes.a_retour]
    marge_par_retour = lignes_ex.marge_perdue_retours.sum() / len(lignes_ex)
    attribuable[t] = {"lignes remisées": len(d_ex), "lignes avec retour en plus": retours_en_plus,
                      "marge perdue par ligne avec retour": marge_par_retour, "marge perdue attribuable": retours_en_plus * marge_par_retour}
attribuable = pd.DataFrame(attribuable).T
attribuable.loc["Ensemble"] = attribuable.sum()
attribuable.loc["Ensemble", "marge perdue par ligne avec retour"] = np.nan
print(f"Surcroît de retours attribuable à la remise, {EXERCICE} (si la remise causait l'écart mesuré) :")
display(attribuable.style.format({"lignes remisées": "{:,.0f}", "lignes avec retour en plus": "{:,.0f}",
                                  "marge perdue par ligne avec retour": "{:,.0f}", "marge perdue attribuable": "{:,.0f}"}, na_rep=""))
tableaux["8_marge_attribuable_remise"] = attribuable

# ---------------------------------------------------------------------------
# 5. Graphique : fréquence selon la promotion et la remise, par canal
# ---------------------------------------------------------------------------
f_statut = statut_canal.frequence.unstack("statut_remise").loc[ORDRE_TYPES, ORDRE_STATUTS]
fig, ax = plt.subplots(figsize=(10, 4.4))
couleurs_statut = [viz.GRIS, viz.CATEGORIES[2], viz.ACCENT]
for i, s in enumerate(ORDRE_STATUTS):
    xs = np.arange(len(ORDRE_TYPES)) + (i - 1) * 0.26
    ax.bar(xs, f_statut[s], width=0.26, color=couleurs_statut[i], label=s)
    for xi, val in zip(xs, f_statut[s]):
        ax.text(xi, val, pct(val), ha="center", va="bottom", fontsize=8.5, color=viz.FONCE)
ax.set_xticks(range(len(ORDRE_TYPES)), ORDRE_TYPES)
ax.set_ylim(0, f_statut.max().max() * 1.25)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
viz.format_pourcent(ax, axe="y")
ax.legend(frameon=False, loc="upper right", ncol=3)
viz.titre(ax, "En magasin, la remise ne change rien aux retours",
          "Part des lignes avec retour selon la promotion et la remise, par canal · trois exercices")
print("Figure enregistrée :", viz.enregistrer(fig, "08_retours_selon_promotion"))
plt.show()

**Lecture**

**Comparaison directe : un lien très faible sur l'ensemble.**

- **Selon le taux annoncé**, aucune progression : 14,6 % sans promotion, 14,3 % à 7 %, 14,6 % à 10 %, 14,5 % à 20 %. Le test rejette H0 (V = 0,006) à cause du seul taux de 5 % (11,1 %), qui ne compte que 1 551 lignes. La corrélation avec le taux de remise effectif est nulle (ρ = +0,007).
- **Les lignes de promotion hors période** (janvier, voir `03_diagnostic`) n'ont pas plus de retours (V = 0,001, H0 conservée).

**Par canal, deux comportements.**

| Fréquence | Hors promotion | En promotion, sans remise | Ligne remisée |
|---|---|---|---|
| En ligne | 17,4 % | 15,0 % | 17,9 % |
| Revendeur | 16,1 % | 14,0 % | 16,5 % |
| Magasins | 12,6 % | 12,6 % | 12,6 % |

**À caractéristiques égales (taille de la ligne, catégorie, gamme, mois, année), la régression logistique confirme.**

| Canal | Rapport de cotes de la remise [intervalle à 95 %] | Probabilité de retour sans remise → avec remise | Rapport de cotes de la taille |
|---|---|---|---|
| En ligne | 1,236 [1,181 ; 1,293] | 15,6 % → 18,6 % (+3,0 points) | 1,535 |
| Revendeur | 1,212 [1,149 ; 1,277] | 14,5 % → 17,1 % (+2,6 points) | 1,507 |
| Magasins | 1,014 [0,986 ; 1,043], p = 0,33 | 12,5 % → 12,7 % (non significatif) | 1,042 |

- **En ligne et chez le revendeur, une ligne remisée a environ 3 points de plus de chances d'avoir un retour**, et le lien résiste au contrôle de la taille, qui joue pourtant fortement dans ces canaux. Le contrôle le renforce même (1,11 sans contrôle, 1,24 avec) : les lignes remisées en ligne ne sont pas les plus grosses.
- **En magasin, la remise ne change rien** : ni sans contrôle (1,005), ni avec (1,014).
- **Le modèle ne sait pas prédire un retour** : AUC de 0,595 en ligne, 0,581 chez le revendeur et 0,510 en magasin (0,5 = hasard), pseudo-R² de 1,5 % au plus. Les liens mesurés sont réels en moyenne, mais ils ne permettent pas de dire à l'avance quelle ligne aura un retour (synthèse, section 12).

**L'enjeu financier est faible.** Si la remise causait tout l'écart mesuré, elle aurait provoqué en 2021 environ 1 448 lignes avec retour de plus (917 en ligne, 473 chez le revendeur, 58 en magasin), soit **0,31 M$ de marge perdue** (0,19 M$ en ligne, 0,10 M$ chez le revendeur, 0,01 M$ en magasin). Pour le levier « supprimer les remises en magasin » de `06` (5,42 M$, `09_q5_remises`), les retours n'ajoutent ni ne retirent rien.

*Interprétation métier (hypothèse)* : en ligne, un achat déclenché par la remise, sans voir le produit, déçoit plus souvent à la réception. Sans motif de retour, on ne peut pas le vérifier : la causalité reste un *indice*.

**Réponse à Q4** : oui, il existe une corrélation positive entre remise et retour, **en ligne et chez le revendeur seulement**, d'environ 3 points de fréquence à caractéristiques égales. Elle est nulle en magasin, et représente au plus 0,31 M$ de marge en 2021.

## 9. Le coût des retours en 2021

La fréquence des retours est presque la même partout. Leur **coût**, lui, dépend du prix de ce qui est retourné, et son **importance** dépend de la rentabilité de celui qui le supporte. On chiffre la situation sur l'exercice 2021, avec 2019 et 2020 pour comparaison.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Le coût des retours par canal et par exercice, face au résultat d'exploitation
# ---------------------------------------------------------------------------
resultat_canal = commun.groupby(["type", "annee"])[["ca_net", "marge_nette", "resultat_exploitation"]].sum()
cout = retours_par(["type", "annee"])[["montant_retour", "marge_perdue", "taux_valeur"]].join(resultat_canal)
total = retours_par("annee")[["montant_retour", "marge_perdue", "taux_valeur"]].join(commun.groupby("annee")[["ca_net", "marge_nette", "resultat_exploitation"]].sum())
total.index = pd.MultiIndex.from_product([["Ensemble"], total.index], names=["type", "annee"])
cout = pd.concat([cout.loc[ORDRE_TYPES], total])
cout["marge perdue ÷ marge nette"] = cout.marge_perdue / cout.marge_nette
cout["marge perdue ÷ résultat d'exploitation"] = cout.marge_perdue / cout.resultat_exploitation
cout["résultat sans aucun retour"] = cout.resultat_exploitation + cout.marge_perdue
print("Coût des retours par canal et par exercice :")
display(cout.style.format({"montant_retour": "{:,.0f}", "marge_perdue": "{:,.0f}", "taux_valeur": "{:.2%}", "ca_net": "{:,.0f}",
                           "marge_nette": "{:,.0f}", "resultat_exploitation": "{:,.0f}", "marge perdue ÷ marge nette": "{:.1%}",
                           "marge perdue ÷ résultat d'exploitation": "{:.1%}", "résultat sans aucun retour": "{:,.0f}"}))
tableaux["9_cout_par_canal"] = cout
cout_ex = cout.xs(EXERCICE, level="annee")
print(f"Marge perdue {EXERCICE} : " + " · ".join(f"{t} {m_dollars(cout_ex.loc[t, 'marge_perdue'], 2)}" for t in ORDRE_TYPES + ["Ensemble"]))

# ---------------------------------------------------------------------------
# 2. D'où vient la baisse de la marge perdue de 2019 à 2021 ? Ventes, taux en valeur ou marge des produits retournés ?
# ---------------------------------------------------------------------------
# marge perdue = chiffre d'affaires brut × taux de retour en valeur × (marge perdue ÷ montant des retours)
decomposition = pd.DataFrame({"chiffre d'affaires brut": par_annee.ca_brut, "taux en valeur": par_annee.taux_valeur,
                              "marge des unités retournées": par_annee.marge_perdue / par_annee.montant_retour,
                              "marge perdue": par_annee.marge_perdue})
evol_decomp = decomposition.loc[EXERCICE] / decomposition.loc[REFERENCE] - 1
print(f"Décomposition de la marge perdue, {REFERENCE} → {EXERCICE} :")
display(pd.concat([decomposition.T, evol_decomp.rename(f"{REFERENCE} → {EXERCICE}")], axis=1)
        .style.format({a: "{:,.4f}" for a in ANNEES} | {f"{REFERENCE} → {EXERCICE}": "{:+.1%}"}))
tableaux["9_decomposition"] = decomposition

# ---------------------------------------------------------------------------
# 3. Magasins ouverts toute l'année, 2021 : les retours font-ils basculer des magasins ?
# ---------------------------------------------------------------------------
mag_ex = (commun[(commun.annee == EXERCICE) & (commun.type == "Magasins") & commun.annee_complete]
          .groupby(["id_boutique", "nom_boutique"])[["ca_net", "montant_retour", "resultat_exploitation"]].sum())
mag_ex["marge_perdue"] = (magasins_lignes[magasins_lignes.annee == EXERCICE].groupby(["id_boutique", "nom_boutique"]).marge_perdue_retours.sum())
mag_ex["résultat sans retour"] = mag_ex.resultat_exploitation + mag_ex.marge_perdue
mag_ex["résultat avec retours divisés par deux"] = mag_ex.resultat_exploitation + mag_ex.marge_perdue / 2
bascule = pd.DataFrame({"magasins bénéficiaires": [(mag_ex[c] > 0).sum() for c in ["resultat_exploitation", "résultat avec retours divisés par deux", "résultat sans retour"]],
                        "résultat des magasins": [mag_ex[c].sum() for c in ["resultat_exploitation", "résultat avec retours divisés par deux", "résultat sans retour"]]},
                       index=["situation 2021", "retours divisés par deux", "aucun retour"])
print(f"Magasins ouverts toute l'année en {EXERCICE} ({len(mag_ex)}) : marge perdue médiane par magasin {m_dollars(mag_ex.marge_perdue.median(), 3)},"
      f" résultat médian {m_dollars(mag_ex.resultat_exploitation.median(), 2)}")
display(bascule.style.format({"résultat des magasins": "{:,.0f}"}))
ecart_au_seuil = (-mag_ex.resultat_exploitation[mag_ex.resultat_exploitation < 0] / mag_ex.marge_perdue[mag_ex.resultat_exploitation < 0])
print(f"Pour un magasin déficitaire, le déficit vaut en médiane {ecart_au_seuil.median():.1f} fois sa marge perdue avec les retours".replace(".", ","))
tableaux["9_magasins_exercice"] = mag_ex
tableaux["9_magasins_bascule"] = bascule

# ---------------------------------------------------------------------------
# 4. Où se concentre le coût en 2021 : catégories, sous-catégories, produits
# ---------------------------------------------------------------------------
ventes_ex = ventes[ventes.annee == EXERCICE]
total_marge_ex = ventes_ex.marge_perdue_retours.sum()


def cout_par(cle, df=ventes_ex):
    t = retours_par(cle, df).sort_values("marge_perdue", ascending=False)
    t["prix_moyen_par_unite"] = df.groupby(cle, observed=True).ca_brut.sum() / df.groupby(cle, observed=True).quantite_vente.sum()
    t["part_de_la_marge_perdue"] = t.marge_perdue / t.marge_perdue.sum()
    t["part_cumulee"] = t.part_de_la_marge_perdue.cumsum()
    t["part_des_lignes_avec_retour"] = df[df.a_retour].groupby(cle, observed=True).size() / df.a_retour.sum()
    return t[["lignes", "frequence", "unites_par_ligne", "prix_moyen_par_unite", "taux_valeur", "marge_perdue",
              "part_de_la_marge_perdue", "part_cumulee", "part_des_lignes_avec_retour"]]


FORMATS_COUT = {"prix_moyen_par_unite": "{:,.0f}", "part_de_la_marge_perdue": "{:.1%}", "part_cumulee": "{:.1%}", "part_des_lignes_avec_retour": "{:.1%}"}
cout_categorie = cout_par("categorie")
print(f"Coût des retours par catégorie, {EXERCICE} :")
montrer(cout_categorie, FORMATS_COUT)
cout_gamme = cout_par("gamme")
print(f"Coût des retours par gamme, {EXERCICE} :")
montrer(cout_gamme, FORMATS_COUT)
cout_sous_categorie = cout_par("sous_categorie")
print(f"Coût des retours, les dix sous-catégories qui pèsent le plus, {EXERCICE} :")
montrer(cout_sous_categorie.head(10), FORMATS_COUT)
cout_categorie_magasins = cout_par("categorie", ventes_ex[ventes_ex.type == "Magasins"])
print(f"Magasins, coût des retours par catégorie, {EXERCICE} :")
montrer(cout_categorie_magasins, FORMATS_COUT)
tableaux["9_cout_categorie"] = cout_categorie
tableaux["9_cout_gamme"] = cout_gamme
tableaux["9_cout_sous_categorie"] = cout_sous_categorie
tableaux["9_cout_categorie_magasins"] = cout_categorie_magasins

rho_freq, _ = stats.spearmanr(cout_sous_categorie.frequence, cout_sous_categorie.marge_perdue)
rho_prix_cout, _ = stats.spearmanr(cout_sous_categorie.prix_moyen_par_unite, cout_sous_categorie.marge_perdue)
print(f"Sur les {len(cout_sous_categorie)} sous-catégories, lien entre la marge perdue et : la fréquence des retours ρ = {rho_freq:+.2f}".replace(".", ","),
      f"· le prix moyen par unité ρ = {rho_prix_cout:+.2f}".replace(".", ","))
cout_produit = cout_par("id_produit")
nb_moitie = int((cout_produit.part_cumulee < 0.5).sum()) + 1
print(f"La moitié de la marge perdue de {EXERCICE} vient de {nb_moitie} produits sur {nombre(len(cout_produit))} ({pct(nb_moitie / len(cout_produit), 0)})")

# ---------------------------------------------------------------------------
# 5. Graphiques : coût par canal sur trois exercices ; fréquence et coût par catégorie en 2021
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 4.4))
mp = cout.marge_perdue.unstack("annee").loc[ORDRE_TYPES]
for i, a in enumerate(ANNEES):
    xs = np.arange(len(ORDRE_TYPES)) + (i - 1) * 0.26
    ax.bar(xs, mp[a], width=0.26, color=COULEURS_ANNEES[a], label=str(a))
    for xi, val in zip(xs, mp[a]):
        ax.text(xi, val, f"{val / 1e6:.1f}".replace(".", ","), ha="center", va="bottom", fontsize=9, color=viz.FONCE)
ax.set_xticks(range(len(ORDRE_TYPES)), ORDRE_TYPES)
viz.format_dollars(ax, axe="y")
ax.legend(frameon=False)
viz.titre(ax, "Les retours des magasins coûtent deux fois moins qu'en 2019, autant qu'en ligne",
          "Marge perdue avec les retours, par canal et par exercice, en millions de dollars")
print("Figure enregistrée :", viz.enregistrer(fig, "08_cout_par_canal"))
plt.show()

cat = cout_categorie.sort_values("marge_perdue")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
ax1.barh(cat.index, cat.frequence, color=viz.CATEGORIES[2])
ax1.axvline(ventes_ex.a_retour.mean(), color=viz.ALERTE, linestyle="--", linewidth=1.5)
ax1.set_xlim(0, 0.2)
ax1.xaxis.set_major_locator(plt.MultipleLocator(0.05))
viz.format_pourcent(ax1, axe="x")
ax1.grid(axis="x"); ax1.grid(axis="y", visible=False)
viz.titre(ax1, "Même fréquence partout", f"Part des lignes avec retour, {EXERCICE}")
ax2.barh(cat.index, cat.marge_perdue, color=viz.PRINCIPALE)
ax2.xaxis.set_major_locator(plt.MultipleLocator(500_000))
viz.format_dollars(ax2, axe="x")
ax2.grid(axis="x"); ax2.grid(axis="y", visible=False)
viz.titre(ax2, "Un coût concentré sur trois catégories", f"Marge perdue avec les retours, {EXERCICE}")
fig.tight_layout()
print("Figure enregistrée :", viz.enregistrer(fig, "08_cout_des_retours"))
plt.show()

**Lecture**

**Ce que coûtent les retours en 2021, canal par canal.**

| 2021 | Marge perdue | Résultat d'exploitation | Marge perdue ÷ résultat | Résultat sans aucun retour |
|---|---|---|---|---|
| En ligne | 2,03 M$ | 99,9 M$ | 2,0 % | 101,9 M$ |
| Revendeur | 1,21 M$ | 52,6 M$ | 2,3 % | 53,8 M$ |
| Magasins | 2,10 M$ | −10,8 M$ | sans objet (déficit) | −8,7 M$ |
| **Ensemble** | **5,34 M$** | **141,7 M$** | **3,8 %** | **147,0 M$** |

- La marge perdue représente de 1,3 % à 1,6 % de la marge nette selon le canal et l'exercice (1,4 % pour l'ensemble en 2021). Son poids dans le résultat de l'entreprise monte de 2,8 % (2019) à 3,8 % (2021), seulement parce que le résultat baisse.
- **Pourquoi la marge perdue baisse plus vite que les ventes** (−33,4 % contre −27,3 %) : le taux en valeur recule de 10,8 % (lignes plus grosses, section 3), tandis que la marge des unités retournées monte un peu (56,7 % → 58,2 % du montant remboursé).

**Les magasins : un coût réel, mais qui n'explique pas le déficit.** Sur les 53 magasins ouverts toute l'année, un magasin perd en médiane 0,038 M$ de marge avec les retours, pour un résultat médian de −0,33 M$ : le déficit d'un magasin déficitaire vaut en médiane **8,9 fois** sa marge perdue.

| Magasins ouverts toute l'année, 2021 | Magasins bénéficiaires | Résultat |
|---|---|---|
| Situation réelle | 4 sur 53 | −11,3 M$ |
| Retours divisés par deux | 4 | −10,2 M$ |
| Aucun retour | 4 | −9,2 M$ |

Ce périmètre exclut Contoso Venezia, ouvert en 2021 (+0,5 M$) : avec lui, les 54 magasins font −10,8 M$, le chiffre de `06`.

Même sans aucun retour, pas un magasin de plus ne deviendrait bénéficiaire. La cause du déficit est ailleurs (hausse des frais généraux, publicité, chute des ventes liée au non-renouvellement de l'assortiment : `06` et `06b`).

**Où se concentre le coût : là où les produits sont chers, pas là où on retourne plus.**

| 2021 | Fréquence | Prix moyen par unité | Marge perdue | Part de la marge perdue | Part des lignes avec retour |
|---|---|---|---|---|---|
| Électroménager (*Home Appliances*) | 14,70 % | 324 $ | 1,93 M$ | 36,1 % | 25,8 % |
| Ordinateurs | 14,47 % | 298 $ | 1,56 M$ | 29,2 % | 28,0 % |
| Appareils photo | 14,63 % | 328 $ | 0,86 M$ | 16,0 % | 14,2 % |
| Autres catégories | 13,7 % à 15,1 % | 29 à 411 $ | 0,99 M$ | 18,6 % | 32,0 % |

- Trois catégories font 81,4 % de la marge perdue, à une fréquence ordinaire. Par gamme, le standard porte 54,8 % du coût, le luxe 36,3 % pour 13,2 % des retours, l'économique 8,9 % pour 34,5 % des retours.
- **Cinq sous-catégories font la moitié du coût** : *Laptops* (0,71 M$), *Washers & Dryers* (0,66 M$), *Camcorders* (0,46 M$), *Projectors & Screens* (0,45 M$), *Refrigerators* (0,42 M$). *Washers & Dryers* a le taux en valeur le plus élevé (3,05 %) : des produits à 1 581 $ vendus par lignes de 5,3 unités, si bien qu'un retour y vaut 3 % de la ligne.
- Sur les 32 sous-catégories, la marge perdue suit le **prix** (ρ = +0,87), presque pas la fréquence (ρ = +0,24). 246 produits sur 2 504 (10 %) font la moitié de la marge perdue de 2021.
- En magasin, les trois mêmes catégories dominent, ordinateurs en tête : ordinateurs (0,73 M$, 34,8 %), électroménager (0,71 M$, 33,8 %), appareils photo (0,31 M$, 14,6 %).

**Ce que la marge perdue ne mesure pas.** Elle suppose que l'unité retournée revient en stock et se revend. Si une partie des unités retournées n'est pas revendable, le coût réel monte, au plus jusqu'au montant remboursé (9,2 M$ en 2021). Les frais de traitement des retours (transport, contrôle) ne sont pas non plus dans nos données. Ces deux points sont à valider avec les gens du métier.

## 10. Le levier « réduire les retours de moitié »

`06_q1_facteurs_rentabilite` (section 8) chiffre ce levier à **2,62 M$** sur l'exercice 2021, net des « autres charges » :

> gain = (marge perdue avec les retours − autres charges sur le montant des retours), toutes boutiques, exercice 2021 ÷ 2

On le recalcule ici à partir des lignes de vente et on vérifie trois points :

1. **la formule** : éviter un retour rend au chiffre d'affaires le montant remboursé et garde la marge, mais les « autres charges » suivent les ventes (`06`, section 8 : un dollar de vente rapporte 0,552 $, et non sa marge brute). Le gain réel est donc la marge perdue **moins** les autres charges sur le chiffre d'affaires conservé. On compare ce gain net à la formule brute (marge perdue ÷ 2) pour mesurer l'écart ;
2. **la répartition par canal** : quelle part du gain revient aux magasins, le seul canal dont le résultat est négatif ;
3. **le ciblage** : s'il existait un segment où les retours sont plus fréquents, il faudrait viser ce segment. Les sections 4 à 8 n'en trouvent pas. Le seul ciblage possible porte sur le **coût** : les familles où un retour évité rapporte le plus.

In [ ]:
# ---------------------------------------------------------------------------
# 1. La formule brute, pour comparaison : marge perdue ÷ 2
# ---------------------------------------------------------------------------
gain_formule_brute = ventes_ex.marge_perdue_retours.sum() / 2
print(f"Formule brute : marge perdue {EXERCICE} ÷ 2 = {m_dollars(ventes_ex.marge_perdue_retours.sum(), 2)} ÷ 2 = {m_dollars(gain_formule_brute, 2)}")

# ---------------------------------------------------------------------------
# 2. Formule nette des charges qui suivent les ventes ("Autres charges"), celle de 06
# ---------------------------------------------------------------------------
reel_ex = reel[reel.annee == EXERCICE]
autres_charges = reel_ex[reel_ex.nom_compte == "Autres charges"].groupby("type").montant.sum()
ca_net_ex = commun[commun.annee == EXERCICE].groupby("type").ca_net.sum()
levier = pd.DataFrame({"marge perdue": ventes_ex.groupby("type").marge_perdue_retours.sum(),
                       "montant des retours": ventes_ex.groupby("type").montant_retour.sum(),
                       "taux des autres charges": autres_charges / ca_net_ex,
                       "résultat d'exploitation": commun[commun.annee == EXERCICE].groupby("type").resultat_exploitation.sum()}).loc[ORDRE_TYPES]
levier["gain net si aucun retour"] = levier["marge perdue"] - levier["taux des autres charges"] * levier["montant des retours"]
PARTS_EVITEES = [0.25, 0.50]
for part in PARTS_EVITEES:
    levier[f"gain si {pct(part, 0)} des retours évités"] = part * levier["gain net si aucun retour"]
levier.loc["Ensemble"] = levier.sum()
levier.loc["Ensemble", "taux des autres charges"] = (autres_charges.sum() / ca_net_ex.sum())
levier["résultat après (50 %)"] = levier["résultat d'exploitation"] + levier[f"gain si {pct(0.5, 0)} des retours évités"]
print(f"Le levier recalculé, par canal, {EXERCICE} :")
display(levier.style.format({c: ("{:.2%}" if c.startswith("taux") else "{:,.0f}") for c in levier.columns}))
tableaux["10_levier_par_canal"] = levier
gain_net = float(levier.loc["Ensemble", f"gain si {pct(0.5, 0)} des retours évités"])
gain_net_magasins = float(levier.loc["Magasins", f"gain si {pct(0.5, 0)} des retours évités"])
resultat_an = float(levier.loc["Ensemble", "résultat d'exploitation"])
print(f"Gain net (moitié des retours évitée) : {m_dollars(gain_net, 2)} (formule brute : {m_dollars(gain_formule_brute, 2)}, "
      f"écart {m_dollars(gain_net - gain_formule_brute, 2, signe=True)}) · hausse du résultat {pct(gain_net / resultat_an, 1, signe=True)}")
resultat_magasins_an = float(levier.loc["Magasins", "résultat d'exploitation"])
print(f"Dont magasins : {m_dollars(gain_net_magasins, 2)} · résultat des magasins {m_dollars(resultat_magasins_an)} → "
      f"{m_dollars(resultat_magasins_an + gain_net_magasins)}")

# ---------------------------------------------------------------------------
# 3. Existe-t-il un segment à cibler par la fréquence ? Les écarts les plus forts trouvés plus haut
# ---------------------------------------------------------------------------
# Pour chaque facteur, l'écart entre le groupe le plus fréquent et la moyenne, et ce que rapporterait
# de ramener ce groupe à la moyenne (un gain "par la fréquence")
ciblage_frequence = []
for nom in ["Sous-catégorie", "Catégorie", "Gamme", "Âge du produit", "Marque", "Promotion (oui ou non)", "Remise sur la ligne (oui ou non)"]:
    colonne = FACTEURS[nom]
    g = ventes_ex.groupby(colonne, observed=True).agg(lignes=("id_vente", "size"), frequence=("a_retour", "mean"),
                                                       marge_perdue=("marge_perdue_retours", "sum"))
    g.index = g.index.map(lambda v: {True: "oui", False: "non"}.get(v, v) if isinstance(v, (bool, np.bool_)) else v)
    haut = g.frequence.idxmax()
    moyenne = ventes_ex.a_retour.mean()
    gain = g.loc[haut, "marge_perdue"] * (1 - moyenne / g.loc[haut, "frequence"]) if g.loc[haut, "frequence"] > moyenne else 0.0
    ciblage_frequence.append({"facteur": nom, "groupe le plus fréquent": str(haut), "fréquence du groupe": g.loc[haut, "frequence"],
                              "fréquence moyenne": moyenne, "gain si ramené à la moyenne": gain})
ciblage_frequence = pd.DataFrame(ciblage_frequence).set_index("facteur").sort_values("gain si ramené à la moyenne", ascending=False)
print(f"Ramener le groupe aux retours les plus fréquents à la moyenne, {EXERCICE} :")
display(ciblage_frequence.style.format({"fréquence du groupe": "{:.1%}", "fréquence moyenne": "{:.1%}", "gain si ramené à la moyenne": "{:,.0f}"}))
tableaux["10_ciblage_par_frequence"] = ciblage_frequence

# ---------------------------------------------------------------------------
# 4. Le ciblage par le coût : viser les sous-catégories où un retour coûte le plus
# ---------------------------------------------------------------------------
cibles = cout_sous_categorie[cout_sous_categorie.part_cumulee.shift(fill_value=0) < 0.5].index.tolist()
ciblees = ventes_ex.sous_categorie.isin(cibles)
part_lignes_retour_ciblees = ventes_ex[ventes_ex.a_retour].sous_categorie.isin(cibles).mean()
marge_ciblee = ventes_ex.loc[ciblees, "marge_perdue_retours"].sum()
taux_ac = float(levier.loc["Ensemble", "taux des autres charges"])
gain_cible = 0.5 * (marge_ciblee - taux_ac * ventes_ex.loc[ciblees, "montant_retour"].sum())
marge_par_retour = {"sous-catégories ciblées": ventes_ex.loc[ciblees & ventes_ex.a_retour, "marge_perdue_retours"].mean(),
                    "autres sous-catégories": ventes_ex.loc[~ciblees & ventes_ex.a_retour, "marge_perdue_retours"].mean()}
print(f"Sous-catégories qui font la moitié de la marge perdue de {EXERCICE} ({len(cibles)}) : {', '.join(cibles)}")
print(f"Elles portent {pct(marge_ciblee / total_marge_ex)} de la marge perdue pour {pct(part_lignes_retour_ciblees)} des lignes avec retour")
print(f"Marge perdue par ligne avec retour : {nombre(marge_par_retour['sous-catégories ciblées'])} $ dans ces sous-catégories, "
      f"{nombre(marge_par_retour['autres sous-catégories'])} $ ailleurs")
print(f"Diviser par deux les retours de ces seules sous-catégories : {m_dollars(gain_cible, 2)} ({pct(gain_cible / gain_net)} du levier, "
      f"en traitant {pct(part_lignes_retour_ciblees)} des retours)")
variantes = pd.DataFrame({
    "gain par an": [gain_formule_brute, gain_net, gain_net_magasins, gain_cible, float(levier.loc["Ensemble", f"gain si {pct(0.25, 0)} des retours évités"])],
    "part des lignes avec retour à traiter": [1, 1, ventes_ex[ventes_ex.a_retour].type.eq("Magasins").mean(), part_lignes_retour_ciblees, 1],
}, index=["Formule brute : marge perdue ÷ 2", "Formule nette des autres charges (celle de 06), moitié des retours",
          "Dont magasins", f"Moitié des retours des {len(cibles)} sous-catégories les plus coûteuses", "Formule nette, un quart des retours"])
variantes["hausse du résultat"] = variantes["gain par an"] / resultat_an
display(variantes.style.format({"gain par an": "{:,.0f}", "part des lignes avec retour à traiter": "{:.1%}", "hausse du résultat": "{:+.2%}"}))
tableaux["10_variantes_du_levier"] = variantes

**Lecture**

**1. La formule : le recalcul retrouve le chiffre de `06`.** Éviter la moitié des retours rend 4,6 M$ de chiffre d'affaires, sur lesquels les « autres charges » (1,09 % du chiffre d'affaires en 2021) sont dues. Le gain net est de **2,62 M$**, le montant retenu par `06`, contre 2,67 M$ avec la formule brute (−0,05 M$), soit +1,8 % de résultat d'exploitation. C'est la même logique que le « ce que rapporte 1 $ de vente » de `06`.

**2. La répartition par canal.**

| 2021 | Gain si la moitié des retours est évitée | Résultat après |
|---|---|---|
| En ligne | 1,00 M$ | 100,8 M$ |
| Revendeur | 0,59 M$ | 53,2 M$ |
| Magasins | 1,03 M$ | −9,7 M$ |
| **Ensemble** | **2,62 M$** | **144,3 M$** |

Les magasins ne recevraient que 1,03 M$, sans qu'aucun ne redevienne bénéficiaire (section 9). Le levier ne peut donc pas être compté en entier pour les magasins.

**3. Pas de segment à cibler par la fréquence.** Ramener à la moyenne le groupe aux retours les plus fréquents de chaque facteur rapporterait au mieux 0,06 M$ (lignes remisées), et moins de 0,02 M$ pour la sous-catégorie (*Fans*, 15,6 %), la marque, la gamme ou l'âge du produit. Aucune cause identifiable dans nos données ne désigne une population où les retours seraient anormalement fréquents.

**4. Un ciblage par le coût.** Les cinq sous-catégories les plus coûteuses portent 50,5 % de la marge perdue pour 22,0 % des lignes avec retour : un retour y coûte 502 $ de marge, contre 139 $ ailleurs. Diviser par deux leurs seuls retours rapporterait **1,32 M$**, la moitié du levier, en ne traitant qu'un retour sur cinq.

| Variante, 2021 | Gain par an | Retours à traiter | Hausse du résultat |
|---|---|---|---|
| Formule brute : marge perdue ÷ 2 | 2,67 M$ | 100 % | +1,88 % |
| Formule nette des autres charges (celle de `06`), moitié des retours | 2,62 M$ | 100 % | +1,85 % |
| dont magasins | 1,03 M$ | 37,1 % | +0,73 % |
| Moitié des retours des 5 sous-catégories les plus coûteuses | 1,32 M$ | 22,0 % | +0,93 % |
| Formule nette, un quart des retours | 1,31 M$ | 100 % | +0,92 % |

**Cohérence avec `06`.**

- **La valeur concorde** : `06` retient 2,62 M$, avec la même formule nette des autres charges ; le recalcul de ce notebook donne le même montant. Le levier est ainsi chiffré comme les autres leviers de `06`, avec ce que rapporte 1 $ de vente.
- **Ce que nos mesures précisent** : aucune cause de fréquence n'est identifiable dans nos données (produit, gamme, âge, magasin, période) ; le coût se concentre sur les produits chers : cinq sous-catégories, 22 % des retours, portent la moitié du gain.
- **La cible de 50 %** n'est soutenue par aucune mesure : on ne connaît pas d'action qui diviserait les retours par deux. C'est un ordre de grandeur, au plus 2,62 M$ ; une cible de 25 % donnerait 1,31 M$.
- **Le levier reste mineur** (+1,8 % de résultat), derrière l'assortiment (46,1 M$), les frais généraux (9,2 M$), la publicité des magasins (6,2 M$) et les remises en magasin (5,42 M$) (`06`, section 8).

## 11. Registre des tests

In [ ]:
verdict_tests = tableau_tests(registre, alpha=ALPHA)
print(f"{len(verdict_tests)} tests · seuil corrigé de Bonferroni : {ALPHA / len(verdict_tests):.5f}".replace(".", ","))
pd.set_option("display.max_colwidth", 140)
display(verdict_tests.style.format({"n": "{:,.0f}", "p": FORMAT_P, "seuil corrigé": "{:.5f}"}).hide(axis="index"))
print("Décisions :", verdict_tests["décision"].value_counts().to_dict())
tableaux["11_registre_des_tests"] = verdict_tests

**Lecture du registre**

- **36 tests**, seuil corrigé de 0,00139 : 15 rejettent H0, 21 la conservent.
- **Les rejets ont presque tous une taille d'effet très faible** (V de 0,004 à 0,085, ρ de +0,007 à +0,110). Ce sont des écarts réels mais sans importance pratique, le cas exact où la valeur p seule tromperait. Le pays des magasins passe tout juste le seuil (p = 0,0013).
- **Deux résultats ont une portée** : le lien entre taille de la ligne et retour, en ligne et chez le revendeur, et le lien entre remise et retour dans ces deux mêmes canaux (rapports de cotes de 1,24 et 1,21).
- **Les tests où H0 est conservée** (produit, âge du produit, gamme, promotion, prix, mois, année, sévérité du COVID-19, stabilité des magasins, remise en magasin, promotions hors période) confirment que la fréquence des retours ne dépend pas de ces facteurs.

## 12. Synthèse : réponses aux questions Q3 et Q4

In [ ]:
# ---------------------------------------------------------------------------
# Registre des constats : chaque constat, la mesure qui le fonde, et son verdict
# ---------------------------------------------------------------------------
f = par_annee.frequence
fc = par_canal.frequence
reg = regression["rapport de cotes"]
constat("Un retour est fréquent mais petit : une ligne sur sept, presque toujours une seule unité",
        f"fréquence {pct(FREQUENCE_GLOBALE)} ; une seule unité dans {pct(taille.loc[1, 'part des lignes avec retour'])} des lignes avec retour", "retenu")
constat("La fréquence des retours est stable sur les trois exercices",
        f"{pct(f[REFERENCE])} · {pct(f[BASCULE])} · {pct(f[EXERCICE])}", "retenu")
constat("La baisse du taux de retour en unités traduit une amélioration",
        f"{pct(par_annee.taux_unites[REFERENCE], 2)} → {pct(par_annee.taux_unites[EXERCICE], 2)}, mais lignes de {dec(par_annee.unites_par_ligne[REFERENCE], 1, False)} à {dec(par_annee.unites_par_ligne[EXERCICE], 1, False)} unités", "non soutenu")
constat("Les retours sont plus fréquents en ligne et chez le revendeur qu'en magasin",
        f"{EXERCICE} : en ligne {pct(fc[('En ligne', EXERCICE)])}, revendeur {pct(fc[('Revendeur', EXERCICE)])}, magasins {pct(fc[('Magasins', EXERCICE)])}", "retenu")
constat("En ligne et chez le revendeur, la fréquence monte avec la taille de la ligne ; pas en magasin",
        f"en ligne {pct(frequence_taille.loc['1 à 5', 'En ligne'])} → {pct(frequence_taille.loc['21 à 50', 'En ligne'])} ; magasins {pct(frequence_taille['Magasins'].min())} à {pct(frequence_taille['Magasins'].max())}", "retenu")
constat("Pourquoi la fréquence suit la taille des lignes en ligne et pas en magasin",
        "une ligne est un cumul journalier, on ne voit ni les commandes ni les tickets", "ouvert")
constat("Une catégorie, une marque, une sous-catégorie ou une période concentre les retours",
        f"V de Cramér de {dec(poids.loc[['Catégorie', 'Marque', 'Sous-catégorie', 'Mois', 'Jour de la semaine'], 'V de Cramér'].min(), signe=False)} à {dec(poids.loc[['Catégorie', 'Marque', 'Sous-catégorie', 'Mois', 'Jour de la semaine'], 'V de Cramér'].max(), signe=False)}", "non soutenu")
constat("Un produit ancien est plus souvent retourné qu'une nouveauté de l'année",
        f"écart de {points(age_annee.frequence.xs(EXERCICE, level='annee').diff().iloc[-1], 2)} en {EXERCICE}, H0 conservée les trois exercices", "non soutenu")
constat("L'entrée de gamme est plus souvent retournée",
        f"{EXERCICE} : économique {pct(gamme_annee.frequence[('Économique', EXERCICE)])}, standard {pct(gamme_annee.frequence[('Standard', EXERCICE)])}, luxe {pct(gamme_annee.frequence[('Luxe', EXERCICE)])}", "non soutenu")
constat("Certains produits sont à problème",
        f"{len(extremes_p)} produits au-delà de 3 écarts-types pour {attendus_p:.0f} attendus ; écart-type des z {dec(z_produits.z.std(), 2, False)}", "non soutenu")
constat("Certains magasins ont durablement plus de retours que les autres",
        f"corrélation des scores z {REFERENCE} et {EXERCICE} : {dec(rho_stab, 2)}", "non soutenu")
constat("La sévérité locale du COVID-19 joue sur les retours d'un magasin",
        f"ρ = {dec(rho_covid)} à mois égal, 2020-2021", "non soutenu")
constat("Les produits chers sont plus souvent retournés", f"ρ = {dec(rho_prix)} par ligne, {dec(rho_produit)} par produit", "non soutenu")
constat("Le coût des retours se concentre sur les produits chers",
        f"{len(cibles)} sous-catégories : {pct(marge_ciblee / total_marge_ex)} de la marge perdue pour {pct(part_lignes_retour_ciblees)} des retours ; ρ prix et coût {dec(rho_prix_cout, 2)}", "retenu")
constat("En ligne et chez le revendeur, une ligne remisée a plus souvent un retour, à caractéristiques égales",
        f"rapport de cotes {dec(reg[('En ligne', 'remise, à caractéristiques égales')], 2, False)} et {dec(reg[('Revendeur', 'remise, à caractéristiques égales')], 2, False)}", "retenu")
constat("C'est la remise qui provoque ce surcroît de retours",
        "lien contrôlé, mais sans motif de retour dans les données", "indice")
constat("En magasin, la remise va avec plus de retours",
        f"rapport de cotes {dec(reg[('Magasins', 'remise, à caractéristiques égales')], 3, False)}, H0 conservée", "non soutenu")
constat("Les retours font basculer des magasins dans le déficit",
        f"magasins bénéficiaires en {EXERCICE} : {bascule.iloc[0, 0]} avec les retours, {bascule.iloc[-1, 0]} sans aucun retour", "non soutenu")
constat("On peut prédire le retour d'une ligne avec les données disponibles",
        f"AUC {dec(qualite.loc['Toutes boutiques', 'AUC'], 3, False)} (magasins {dec(qualite.loc['Magasins', 'AUC'], 3, False)})", "non soutenu")
registre_constats = pd.DataFrame(constats)
pd.set_option("display.max_colwidth", 160)
display(registre_constats.style.hide(axis="index"))
tableaux["12_registre_des_constats"] = registre_constats.set_index("constat")

### Message principal : les retours et les remises ne sont pas les problèmes prioritaires

Sur l'exercice 2021, l'entreprise dégage 141,7 M$ de résultat d'exploitation et les magasins perdent 10,8 M$. Les retours coûtent 5,34 M$ de marge, et en éviter la moitié rapporterait 2,62 M$ (+1,8 % de résultat). Le surcroît de retours lié aux remises pèse au plus 0,31 M$. Ces montants sont **bien plus faibles** que ceux des trois problèmes établis par `06_q1_facteurs_rentabilite` et `06b_q1_chute_des_ventes` : la hausse des frais généraux, le non-renouvellement de l'assortiment et la perte des grands comptes de la boutique en ligne.

| Problème | Enjeu chiffré | Ce que mesure le chiffre | Source |
|---|---|---|---|
| Perte des grands comptes de la boutique en ligne | 169,0 M$ (2019) → 86,6 M$ (2020) → 0,1 M$ (2021) | ventes aux entreprises clientes historiques, mesurées sur la table des ventes en ligne détaillée : **ne s'additionne pas** avec les autres montants du tableau | `06b` |
| Non-renouvellement de l'assortiment | 46,1 M$ | levier : revenir au rythme de lancement de 2019 (552, 429 puis 272 lancements) ; les nouveautés de l'année portent 69 % de la chute des ventes des magasins | `06`, `06b` |
| Hausse des frais généraux | +26,3 M$ en deux exercices (155,0 → 181,3 M$), dont +12,9 M$ en magasin | budget augmenté par décision de gestion ; le levier « personnel, administration, informatique au montant de 2019 » vaut 9,2 M$ | `06` |
| Publicité des magasins | 6,2 M$ | levier : budget réduit de 10 % | `06` |
| Remises en magasin | 5,42 M$ | levier : supprimer les remises en magasin | `06`, `09_q5_remises` |
| Retours | 5,34 M$ de marge perdue ; levier 2,62 M$ | levier : éviter la moitié des retours | ce notebook |
| Surcroît de retours lié aux remises | 0,31 M$ au plus | marge perdue attribuable si la remise causait tout l'écart mesuré | ce notebook |

Les leviers sur les retours (et ceux sur les remises, `07_q2_promotions` et `09_q5_remises`) restent valables, mais **secondaires** : ils ne redressent ni le résultat de l'entreprise ni celui des magasins.

### Réponse à Q3 : quels sont les patterns de retour de produits ?

Il n'y a pas de « pattern » au sens d'un produit, d'une catégorie, d'une gamme, d'une boutique ou d'une période qui concentrerait les retours.

| Question | Réponse | Verdict |
|---|---|---|
| À quoi ressemble un retour ? | Une ligne sur sept (14,5 %), une seule unité dans 97,3 % des cas ; 1,32 % du chiffre d'affaires en 2021 | Retenu |
| A-t-il changé depuis 2019 ? | Fréquence stable (14,4 % → 14,5 %) ; marge perdue 8,0 → 5,8 → 5,3 M$, au rythme des ventes | Retenu |
| Les canaux diffèrent-ils ? | Oui : 16,7 % en ligne, 15,1 % chez le revendeur, 12,4 % en magasin en 2021, surtout par la taille des lignes | Retenu |
| Les produits anciens, l'entrée de gamme ? | Non, ni en fréquence ni en coût, sur aucun exercice | Non soutenu |
| Des produits ou des magasins à problème ? | Non : écarts de hasard pour les produits, écarts non durables pour les magasins | Non soutenu |
| Une saison, l'effet du COVID-19 ? | Non | Non soutenu |
| Où les retours coûtent-ils ? | Sur les produits chers : cinq sous-catégories font la moitié de la marge perdue de 2021 | Retenu |
| Les retours sont-ils un problème prioritaire ? | Non : 5,34 M$ de marge perdue et un levier de 2,62 M$ en 2021, loin derrière les frais généraux, l'assortiment et les grands comptes en ligne (message principal) | Retenu |

### Réponse à Q4 : existe-t-il une corrélation entre les remises et le taux de retour ?

| Question | Réponse | Verdict |
|---|---|---|
| Le taux annoncé joue-t-il ? | Non | Non soutenu |
| Une ligne remisée a-t-elle plus souvent un retour ? | En ligne et chez le revendeur, oui : environ +3 points à caractéristiques égales. En magasin, non | Retenu |
| La remise en est-elle la cause ? | Compatible, sans preuve directe (aucun motif de retour) | Indice |
| Est-ce un enjeu financier ? | Non : au plus 0,31 M$ de marge en 2021, 0,01 M$ en magasin, sans commune mesure avec les problèmes prioritaires (message principal) | Retenu |

### Registre des constats

| Constat | Verdict |
|---|---|
| Un retour est fréquent mais petit : une ligne sur sept, presque toujours une seule unité | Retenu |
| La fréquence des retours est stable sur les trois exercices | Retenu |
| La baisse du taux de retour en unités traduit une amélioration | Non soutenu (effet de la taille des lignes) |
| Les retours sont plus fréquents en ligne et chez le revendeur qu'en magasin | Retenu |
| En ligne et chez le revendeur, la fréquence monte avec la taille de la ligne ; pas en magasin | Retenu |
| Pourquoi la fréquence suit la taille des lignes en ligne et pas en magasin | Ouvert |
| Une catégorie, une marque, une sous-catégorie ou une période concentre les retours | Non soutenu |
| Un produit ancien est plus souvent retourné qu'une nouveauté de l'année | Non soutenu |
| L'entrée de gamme est plus souvent retournée | Non soutenu |
| Certains produits sont à problème | Non soutenu |
| Certains magasins ont durablement plus de retours que les autres | Non soutenu |
| La sévérité locale du COVID-19 joue sur les retours d'un magasin | Non soutenu |
| Les produits chers sont plus souvent retournés | Non soutenu |
| Le coût des retours se concentre sur les produits chers | Retenu |
| En ligne et chez le revendeur, une ligne remisée a plus souvent un retour, à caractéristiques égales | Retenu |
| C'est la remise qui provoque ce surcroît de retours | Indice |
| En magasin, la remise va avec plus de retours | Non soutenu |
| Les retours font basculer des magasins dans le déficit | Non soutenu |
| On peut prédire le retour d'une ligne avec les données disponibles | Non soutenu |

### Recommandations, chiffrées sur l'exercice 2021

1. **Ne pas faire des retours une priorité.** Ils coûtent 5,34 M$ de marge en 2021 (3,8 % du résultat d'exploitation, 1,3 % à 1,4 % de la marge nette selon le canal). Le levier vaut au plus 2,62 M$, loin derrière le renouvellement de l'assortiment (46,1 M$) et les frais généraux (9,2 M$) chiffrés dans `06` (tableau du message principal).
2. **Si une action est menée, la cibler sur le coût et non sur la fréquence** : *Laptops*, *Washers & Dryers*, *Camcorders*, *Projectors & Screens* et *Refrigerators*. Diviser leurs retours par deux rapporterait 1,32 M$ en traitant 22 % des retours (contrôle avant livraison, conseil à l'achat, conditions de reprise des appareils chers).
3. **Ne pas compter sur les retours pour redresser les magasins** : 2,10 M$ de marge perdue en 2021, 1,03 M$ de gain au plus avec la moitié des retours, et aucun magasin de plus bénéficiaire même sans aucun retour.
4. **Ne pas durcir la politique de retour en ligne** : les retours y sont plus fréquents, mais ils coûtent 2,0 % du résultat d'un canal à 99,9 M$. Le surcroît lié aux remises (0,19 M$ en ligne) ne justifie pas, à lui seul, de réduire les remises en ligne ; la décision relève de `09_q5_remises`.
5. **Le levier de `06` est cohérent avec ce notebook** : `06` retient 2,62 M$, avec la même formule nette des autres charges, et le recalcul à partir des lignes de vente donne le même montant. Nos mesures en précisent la lecture : aucune cause de fréquence identifiable dans nos données ; coût concentré sur les produits chers (cinq sous-catégories, 22 % des retours, la moitié du gain). Le gain revient pour 1,03 M$ aux magasins.
6. **Suivre la fréquence et le taux en valeur**, à taille de ligne égale, jamais le taux en unités seul : il a été divisé par deux en deux exercices sans que rien ne change.
7. **Collecter le motif et la date du retour** : sans eux, on ne peut ni distinguer un défaut d'une déception, ni vérifier l'effet de la remise en ligne.

**Constat : prédire un retour ne sert à rien avec ces données.** Avec toutes les variables disponibles (remise, taille de la ligne, canal, catégorie, gamme, mois, année), la régression logistique atteint une AUC de 0,565 toutes boutiques et de 0,510 en magasin, à peine mieux que le hasard (0,5). Ce résultat le démontre : il n'y a pas de motif clair dans les données, les retours se répartissent presque uniformément (sections 4 à 7), et les deux informations qui pourraient en porter un manquent (aucun motif de retour, pas de client). Un modèle plus complexe ne trouverait pas un signal que ces variables ne portent pas.

### Points à valider avec les gens du métier

- La politique de retour est-elle la même dans les trois canaux et dans tous les magasins (délai, conditions) ?
- Les unités retournées sont-elles remises en vente à plein prix ? Si non, le coût réel dépasse la marge perdue, jusqu'à 9,2 M$ en 2021.
- Quels sont les frais de traitement d'un retour (transport, contrôle, reconditionnement), absents de nos données ?
- En ligne et chez le revendeur, que regroupe une ligne de vente (une ou plusieurs commandes du jour) ? C'est la clé du lien entre taille de la ligne et retour.
- Une action ciblée sur les appareils chers peut-elle réduire leurs retours, et de combien ? La cible de 50 % n'est pas mesurée.

### Limites

- **Aucun motif de retour** : on ne sait pas pourquoi un produit est rapporté (défaut, erreur, changement d'avis).
- **Aucune date de retour** : le retour est rattaché à la ligne de vente, on ne mesure pas de délai.
- **Pas de client** : on ne peut pas repérer des acheteurs qui retournent souvent.
- **Une ligne est un cumul journalier** : « une ligne avec retour » signifie qu'au moins une vente du jour, pour ce produit dans cette boutique, a donné lieu à un retour.
- **Corrélation et non causalité** pour Q4, même après contrôle.

## Enregistrement des tableaux

In [ ]:
# Tous les tableaux de ce notebook dans un classeur Excel, un onglet par tableau,
# pour les reprendre dans le rapport d'analyse (livrable L4) sans relancer le code.
tableaux_excel = {}
for onglet, table in tableaux.items():
    table = table.copy()
    # Excel n'accepte ni les périodes ni les intervalles : on les écrit comme du texte
    if isinstance(table.index, (pd.PeriodIndex, pd.IntervalIndex, pd.CategoricalIndex)):
        table.index = table.index.astype(str)
    if isinstance(table.index, pd.MultiIndex):
        table.index = table.index.set_levels([lvl.astype(str) for lvl in table.index.levels])
    if isinstance(table.columns, pd.CategoricalIndex):
        table.columns = table.columns.astype(str)
    tableaux_excel[onglet] = table
chemin_excel = analyse.exporter_tableaux(tableaux_excel, config.L4 / "q3_q4_tableaux.xlsx")
print("Classeur enregistré :", chemin_excel)
print("Onglets :", list(tableaux_excel))